# Phase 2 — EDA chi tiết: LVVO (D-A), COCO subset (D-B), DanceTrack (T)

Notebook này phân tích **đúng các cohort đã đóng băng** ở lần chạy `det-track-qualification-261026-v3` (hash GT được kiểm lại trước khi dùng). Không chạy model, không chấm điểm. Mục tiêu: hiểu dữ liệu đủ để thiết kế đánh giá ở phase 3/4 và biết trước các giới hạn thống kê.

**Nguyên tắc khoa học áp dụng**
- Đơn vị phụ thuộc được xử lý đúng: LVVO theo **nhóm nguồn** (và nhóm gộp qua cặp gần trùng), DanceTrack theo **sequence**. Báo *design effect* và *effective n*; khoảng tin cậy dùng bootstrap theo cụm.
- So sánh train/val của DanceTrack là kiểm định ở mức sequence (Mann–Whitney, KS) kèm **Cliff's δ**, hiệu chỉnh **Holm** cho nhiều đặc trưng; KS gộp chỉ để mô tả.
- Ảnh gần trùng (dHash) được đo lại bằng **slide Jaccard** (cắt viền đen, chỉ so vùng slide ở giữa, so mặt nạ nét vẽ). NCC thô và ink-overlap toàn khung bị loại vì các frame cùng bài giảng chia sẻ nền trắng, viền đen và cửa sổ giao diện nên điểm cao dù nội dung khác (đã thấy trên lưới ảnh hiệu chuẩn). Mọi độ đo tự động ở đây chỉ là **cờ sàng lọc**; kết luận trùng lặp cần người duyệt lưới ảnh. Không tự loại ảnh nào.
- Ngưỡng cờ ("ít hỗ trợ", "dịch chuyển phân phối", ...) là quy tắc heuristic được in rõ trong bảng quan sát, không phải kết luận suy diễn.
- Mô tả ≠ suy diễn: mọi so sánh với toàn bộ COCO val2017 là **kiểm tra tính đại diện có chủ đích** (tập con được phân tầng theo lớp/kích thước nên lệch có chủ ý).

**Cách chạy:** chạy tuần tự trên Colab (CPU đủ). Cần thư mục `MyDrive/DetTrack/det-track-qualification-261026-v3/` từ lần chạy trước. DanceTrack được đọc bằng HTTP range (chỉ GT và vài frame mẫu, không tải 11 GB). Kết quả: `MyDrive/DetTrack/<RUN_ID>/` (hình, bảng CSV, `eda-observations.json`, `bundle.zip`).

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, subprocess, time, shutil, zipfile, re, platform, traceback
from datetime import datetime, timezone

IN_COLAB = Path('/content').exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
OFFLINE = os.environ.get('DT_OFFLINE') == '1'

RUN_ID = 'det-track-eda-261026-v2'
WORK = Path('/content/dt-eda') if IN_COLAB else Path.cwd() / 'dt-eda-work'
PERSIST = Path('/content/drive/MyDrive/DetTrack') if IN_COLAB else WORK / 'persistent'
PRIOR = Path(os.environ.get('DT_PRIOR', PERSIST / 'det-track-qualification-261026-v3'))
OUT = PERSIST / RUN_ID
for p in (WORK, OUT):
    p.mkdir(parents=True, exist_ok=True)
if not PRIOR.exists():
    raise FileNotFoundError(f'Thiếu {PRIOR}: chạy notebook qualification trước (hoặc đặt PRIOR).')
if not OFFLINE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'gdown', 'requests', 'scipy'], check=True)
print('Frozen inputs:', PRIOR, '\nResults:', OUT)

In [ ]:
SOURCES = {'experiments/scripts/detection_tracking_qualification.py': '"""Dataset qualification helpers for detection (03a) and tracking (03b).\n\nPure functions so the Colab notebook and the unit tests share one implementation.\nNothing here downloads data or runs a model.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nimport zipfile\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\nimport numpy as np\n\n# Frozen by the claim-freeze decision report (2026-10-26).\nCOCO_TAXONOMY = {  # COCO category id -> name\n    1: "person", 44: "bottle", 62: "chair", 72: "tv", 73: "laptop",\n    74: "mouse", 76: "keyboard", 77: "cell phone", 84: "book", 85: "clock",\n}\nPRACTICAL_THRESHOLDS = {"delta_map_50_95": 0.02, "delta_hota": 2.0}\nBOOTSTRAP = {"resamples": 10000, "ci": 0.95, "method": "paired_percentile"}\nCOCO_SIZE_EDGES = (32 ** 2, 96 ** 2)  # COCO small/medium/large area edges\n\n\n# --------------------------------------------------------------------------\n# Hashing, archives\n# --------------------------------------------------------------------------\ndef sha256_file(path: Path, chunk: int = 1 << 20) -> str:\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(chunk), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef canonical_json_sha256(value: Any) -> str:\n    return hashlib.sha256(\n        json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")\n    ).hexdigest()\n\n\ndef safe_extract(archive: Path, destination: Path,\n                 keep: Callable[[str], bool] = lambda name: True) -> list[str]:\n    """Extract members accepted by `keep`; reject traversal and symlinks. CRC checked."""\n    destination = Path(destination).resolve()\n    extracted: list[str] = []\n    with zipfile.ZipFile(archive) as zf:\n        for info in zf.infolist():\n            name = info.filename\n            if info.is_dir() or name.startswith("__MACOSX/") or name.endswith(".DS_Store"):\n                continue\n            if not keep(name):\n                continue\n            target = (destination / name).resolve()\n            if not target.is_relative_to(destination):\n                raise ValueError(f"Unsafe archive path: {name}")\n            if (info.external_attr >> 16) & 0o170000 == 0o120000:\n                raise ValueError(f"Symlink in archive: {name}")\n            if target.exists() and target.stat().st_size == info.file_size:\n                extracted.append(name)\n                continue\n            target.parent.mkdir(parents=True, exist_ok=True)\n            with zf.open(info) as src, target.open("wb") as dst:\n                dst.write(src.read())\n            extracted.append(name)\n    return extracted\n\n\ndef reconcile(raw: int, eligible: int, excluded: int, unresolved: int) -> dict[str, int]:\n    """Enforce raw = eligible + excluded + unresolved; never drop units silently."""\n    if raw != eligible + excluded + unresolved:\n        raise ValueError(\n            f"Denominator mismatch: raw={raw} eligible={eligible} "\n            f"excluded={excluded} unresolved={unresolved}")\n    return dict(raw=raw, eligible=eligible, excluded=excluded, unresolved=unresolved)\n\n\ndef exact_overlap(left: Iterable[str], right: Iterable[str]) -> list[str]:\n    return sorted(set(left) & set(right))\n\n\n# --------------------------------------------------------------------------\n# Box helpers\n# --------------------------------------------------------------------------\ndef size_stratum(area: float) -> str:\n    small, medium = COCO_SIZE_EDGES\n    return "small" if area < small else "medium" if area < medium else "large"\n\n\ndef iou_matrix_xywh(boxes: np.ndarray) -> np.ndarray:\n    """Pairwise IoU for an (n,4) xywh array; diagonal set to 0."""\n    n = len(boxes)\n    if n == 0:\n        return np.zeros((0, 0))\n    x1, y1 = boxes[:, 0], boxes[:, 1]\n    x2, y2 = x1 + boxes[:, 2], y1 + boxes[:, 3]\n    iw = np.clip(np.minimum(x2[:, None], x2[None]) - np.maximum(x1[:, None], x1[None]), 0, None)\n    ih = np.clip(np.minimum(y2[:, None], y2[None]) - np.maximum(y1[:, None], y1[None]), 0, None)\n    inter = iw * ih\n    area = boxes[:, 2] * boxes[:, 3]\n    union = area[:, None] + area[None] - inter\n    iou = np.divide(inter, union, out=np.zeros_like(inter, dtype=float), where=union > 0)\n    np.fill_diagonal(iou, 0.0)\n    return iou\n\n\n# --------------------------------------------------------------------------\n# Detection A: LVVO -> COCO\n# --------------------------------------------------------------------------\ndef lvvo_to_coco(dataset_root: Path) -> dict[str, Any]:\n    """Convert the released LVVO 1k four-class labels to COCO JSON.\n\n    Image ids are 1-based over sorted file stems (stable). Invalid boxes and\n    unreadable pairs go to `excluded`, never silently dropped. Images with zero\n    valid boxes stay in the image list as valid negatives.\n    """\n    from PIL import Image\n\n    root = Path(dataset_root)\n    meta = json.loads((root / "dataset_info.json").read_text(encoding="utf-8"))\n    categories = [{"id": int(v), "name": k} for k, v in sorted(meta["categories"].items(), key=lambda kv: int(kv[1]))]\n    class_names = {c["id"]: c["name"] for c in categories}\n    image_paths = {p.stem: p for p in (root / "images").glob("*.jpg")}\n    label_paths = {p.stem: p for p in (root / "labels").glob("*.json")}\n    images, annotations, excluded, ledger = [], [], [], []\n    next_ann = 1\n    for index, stem in enumerate(sorted(image_paths.keys() | label_paths.keys()), start=1):\n        ip, lp = image_paths.get(stem), label_paths.get(stem)\n        if ip is None or lp is None:\n            excluded.append(dict(stem=stem, reason="missing_pair"))\n            continue\n        try:\n            with Image.open(ip) as im:\n                im.load()\n                width, height = im.size\n            labels = json.loads(lp.read_text(encoding="utf-8"))\n        except Exception as exc:  # decode/parse failure is counted, not hidden\n            excluded.append(dict(stem=stem, reason=f"read_error:{type(exc).__name__}"))\n            continue\n        valid_boxes = 0\n        for obj in labels.get("objects", []):\n            b = obj.get("boundingBox", {})\n            x1, y1, x2, y2 = (b.get(k) for k in ("xmin", "ymin", "xmax", "ymax"))\n            numeric = all(isinstance(v, (int, float)) for v in (x1, y1, x2, y2))\n            cid = obj.get("class")\n            if not (numeric and 0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height) or cid not in class_names:\n                excluded.append(dict(stem=stem, reason="invalid_box_or_class", box=[x1, y1, x2, y2], cls=cid))\n                continue\n            w, h = x2 - x1, y2 - y1\n            annotations.append(dict(id=next_ann, image_id=index, category_id=int(cid),\n                                    bbox=[float(x1), float(y1), float(w), float(h)],\n                                    area=float(w * h), iscrowd=0))\n            next_ann += 1\n            valid_boxes += 1\n        images.append(dict(id=index, file_name=ip.name, width=width, height=height))\n        inst = re.match(r"i(\\d+)_", stem)\n        course = re.search(r"_c(\\d+)_", stem)\n        video = re.search(r"_v(\\d+)_", stem)\n        ledger.append(dict(image_id=index, stem=stem, file_name=ip.name, width=width, height=height,\n                           n_boxes=valid_boxes,\n                           group="/".join(f"{k}{m.group(1)}" for k, m in\n                                          (("i", inst), ("c", course), ("v", video)) if m),\n                           sha256=sha256_file(ip)))\n    return dict(coco=dict(images=images, annotations=annotations, categories=categories),\n                excluded=excluded, ledger=ledger, class_names=class_names,\n                stems_total=len(image_paths.keys() | label_paths.keys()),\n                released_images=len(image_paths), released_labels=len(label_paths))\n\n\ndef dhash64(path: Path) -> int:\n    from PIL import Image\n    with Image.open(path) as im:\n        arr = np.asarray(im.convert("L").resize((9, 8), Image.Resampling.LANCZOS))\n    bits = (arr[:, 1:] > arr[:, :-1]).ravel()\n    value = 0\n    for bit in bits:\n        value = (value << 1) | int(bit)\n    return value\n\n\ndef near_duplicate_pairs(hashes: dict[str, int], groups: dict[str, str],\n                         sha: dict[str, str], max_distance: int = 4) -> list[dict[str, Any]]:\n    """All non-identical pairs within dHash Hamming distance; review flags only."""\n    keys = sorted(hashes)\n    values = np.array([hashes[k] for k in keys], dtype=np.uint64)\n    pairs = []\n    for i in range(len(keys) - 1):\n        xor = values[i] ^ values[i + 1:]\n        dist = np.array([int(v).bit_count() for v in xor])\n        for j in np.nonzero(dist <= max_distance)[0]:\n            other = keys[i + 1 + int(j)]\n            if sha[keys[i]] == sha[other]:\n                continue\n            pairs.append(dict(left=keys[i], right=other, distance=int(dist[j]),\n                              same_group=groups[keys[i]] == groups[other]))\n    return pairs\n\n\n# --------------------------------------------------------------------------\n# Detection B: COCO subset\n# --------------------------------------------------------------------------\ndef select_coco_subset(coco: dict[str, Any], n_images: int, seed: int,\n                       taxonomy: dict[int, str] = COCO_TAXONOMY) -> dict[str, Any]:\n    """Seeded subset: per-class quota of images, balanced over box-size strata, then fill.\n\n    Eligible image = at least one non-crowd taxonomy box. Result is deterministic\n    for (coco, n_images, seed).\n    """\n    rng = np.random.default_rng(seed)\n    by_image: dict[int, list[dict]] = {}\n    for ann in coco["annotations"]:\n        if ann["category_id"] in taxonomy and not ann.get("iscrowd", 0):\n            by_image.setdefault(ann["image_id"], []).append(ann)\n    eligible = sorted(by_image)\n    # (class, stratum) -> sorted candidate image ids\n    cell: dict[tuple[int, str], list[int]] = {}\n    for image_id in eligible:\n        for ann in by_image[image_id]:\n            cell.setdefault((ann["category_id"], size_stratum(ann["area"])), []).append(image_id)\n    cell = {k: sorted(set(v)) for k, v in cell.items()}\n    class_images = {c: sorted({i for (cc, _), v in cell.items() if cc == c for i in v}) for c in taxonomy}\n    quota = max(1, n_images // len(taxonomy))\n    selected: set[int] = set()\n    for cid in sorted(taxonomy, key=lambda c: (len(class_images[c]), c)):  # rarest first\n        members = set(class_images[cid])\n        have = lambda: len(selected & members)\n        strata = [s for s in ("small", "medium", "large") if cell.get((cid, s))]\n        pools = {s: list(rng.permutation(cell[(cid, s)])) for s in strata}\n        while strata and have() < quota and len(selected) < n_images:\n            for s in list(strata):\n                while pools[s] and pools[s][-1] in selected:\n                    pools[s].pop()\n                if not pools[s]:\n                    strata.remove(s)\n                    continue\n                selected.add(int(pools[s].pop()))\n                if have() >= quota or len(selected) >= n_images:\n                    break\n    remaining = [i for i in eligible if i not in selected]\n    need = min(n_images, len(eligible)) - len(selected)\n    if need > 0:\n        selected.update(int(i) for i in rng.permutation(remaining)[:need])\n    ids = sorted(selected)\n    keep = set(ids)\n    anns = [a for a in coco["annotations"] if a["image_id"] in keep and a["category_id"] in taxonomy]\n    return dict(image_ids=ids, annotations=anns,\n                eligible_images=len(eligible), seed=seed, n_images=n_images,\n                quota_per_class=quota)\n\n\n# --------------------------------------------------------------------------\n# Tracking: MOT-format GT\n# --------------------------------------------------------------------------\nMOT_MEMBER = re.compile(\n    r"(?:^|.*/)(?P<split>train|val|test)/(?P<seq>[^/]+)/"\n    r"(?P<kind>seqinfo\\.ini|gt/gt\\.txt|img1/(?P<frame>\\d+)\\.jpg)$")\n\n\n_MEMBER_TAIL = re.compile(\n    r"(?:^|.*/)(?P<seq>[^/]+)/(?P<kind>seqinfo\\.ini|gt/gt\\.txt|img1/(?P<frame>\\d+)\\.jpg)$")\n_SPLIT_DIR = re.compile(r"^(train|val|test)\\d*$", re.I)\n\n\ndef parse_mot_member(name: str, default_split: str | None = None) -> dict[str, str] | None:\n    """Classify an archive member of a MOT-style dataset.\n\n    The split is the nearest ancestor directory named train*/val*/test* (so `train1/seq/...`\n    counts as train), else `default_split` (typically taken from the archive file name).\n    """\n    m = _MEMBER_TAIL.match(name)\n    if not m or m["seq"] in (".", ".."):\n        return None\n    dirs = name.split("/")\n    ancestors = dirs[: -(2 if m["kind"] == "seqinfo.ini" else 3)]  # drop seq dir (+ gt/img1) and file\n    split = next((_SPLIT_DIR.match(p).group(1).lower() for p in reversed(ancestors) if _SPLIT_DIR.match(p)),\n                 default_split)\n    if split is None:\n        return None\n    return dict(split=split, seq=m["seq"], kind=m["kind"], frame=m["frame"])\n\n\ndef parse_seqinfo(text: str) -> dict[str, str]:\n    out = {}\n    for line in text.splitlines():\n        if "=" in line:\n            key, value = line.split("=", 1)\n            out[key.strip()] = value.strip()\n    return out\n\n\ndef sample_frame_indices(seq_length: int, pairs: int = 8) -> list[int]:\n    """Frame numbers (1-based): first, last, and `pairs` consecutive pairs spread over the video."""\n    if seq_length <= 0:\n        return []\n    picks = {1, seq_length}\n    for k in range(pairs):\n        t = 1 + int(round(k * max(seq_length - 2, 0) / max(pairs, 1)))\n        picks.update({t, min(t + 1, seq_length)})\n    return sorted(picks)\n\n\ndef parse_mot_gt(text: str) -> np.ndarray:\n    rows = [r for r in (line.strip().split(",") for line in text.splitlines()) if len(r) >= 6]\n    if not rows:\n        return np.zeros((0, 9))\n    width = max(len(r) for r in rows)\n    arr = np.full((len(rows), max(width, 9)), np.nan)\n    for i, r in enumerate(rows):\n        arr[i, :len(r)] = [float(v) for v in r]\n    return arr\n\n\ndef mot_sequence_stats(gt: np.ndarray, seq_length: int, im_width: int, im_height: int) -> dict[str, Any]:\n    """Per-sequence EDA: density, track lengths, ID integrity, occlusion proxy."""\n    if len(gt) == 0:\n        return dict(boxes=0, tracks=0, issues=["empty_gt"])\n    frames, ids = gt[:, 0].astype(int), gt[:, 1].astype(int)\n    boxes = gt[:, 2:6]\n    dup_frame_id = len(gt) - len({(f, i) for f, i in zip(frames, ids)})\n    bad_size = int(((boxes[:, 2] <= 0) | (boxes[:, 3] <= 0)).sum())\n    out_of_range = int(((frames < 1) | (frames > seq_length)).sum())\n    off_image = int(((boxes[:, 0] + boxes[:, 2] < 0) | (boxes[:, 1] + boxes[:, 3] < 0) |\n                     (boxes[:, 0] > im_width) | (boxes[:, 1] > im_height)).sum())\n    track_len, gap_tracks = [], 0\n    for tid in np.unique(ids):\n        f = np.sort(frames[ids == tid])\n        track_len.append(len(f))\n        if f[-1] - f[0] + 1 != len(f):\n            gap_tracks += 1\n    per_frame = np.bincount(frames[(frames >= 1) & (frames <= seq_length)], minlength=seq_length + 1)[1:]\n    occluded = 0\n    for f in np.unique(frames):\n        b = boxes[frames == f]\n        if len(b) > 1:\n            occluded += int((iou_matrix_xywh(b).max(axis=1) > 0.3).sum())\n    conf = gt[:, 6] if gt.shape[1] > 6 else np.ones(len(gt))\n    return dict(\n        boxes=int(len(gt)), tracks=int(len(track_len)), frames_with_boxes=int((per_frame > 0).sum()),\n        seq_length=int(seq_length), boxes_per_frame_mean=float(per_frame.mean()),\n        boxes_per_frame_max=int(per_frame.max()),\n        track_len_min=int(min(track_len)), track_len_median=float(np.median(track_len)),\n        track_len_max=int(max(track_len)), tracks_with_gaps=int(gap_tracks),\n        duplicate_frame_id_rows=int(dup_frame_id), nonpositive_size_rows=bad_size,\n        frame_out_of_range_rows=out_of_range, off_image_rows=off_image,\n        ignored_conf0_rows=int((np.nan_to_num(conf, nan=1.0) == 0).sum()),\n        occluded_box_fraction=float(occluded / len(gt)),\n        box_height_median=float(np.median(boxes[:, 3])),\n        ids_valid=bool((ids >= 0).all() and (gt[:, 1] == ids).all()),  # DanceTrack ids start at 0\n    )\n\n\ndef mot_blocking_issues(stats: dict[str, Any]) -> int:\n    return int(stats.get("duplicate_frame_id_rows", 0) + stats.get("nonpositive_size_rows", 0)\n               + stats.get("frame_out_of_range_rows", 0) + (0 if stats.get("ids_valid", True) else 1)\n               + (1 if "empty_gt" in stats.get("issues", []) else 0))\n\n\ndef camera_motion_pixels(gray_pairs: list[tuple[np.ndarray, np.ndarray]]) -> float:\n    """Median global shift (px) between consecutive frames via phase correlation.\n\n    A descriptive proxy for static vs moving camera, not a ground-truth label.\n    """\n    import cv2\n\n    shifts = []\n    for a, b in gray_pairs:\n        (dx, dy), _ = cv2.phaseCorrelate(np.float32(a), np.float32(b))\n        shifts.append(float(np.hypot(dx, dy)))\n    return float(np.median(shifts)) if shifts else float("nan")\n\n\n# --------------------------------------------------------------------------\n# Decision\n# --------------------------------------------------------------------------\ndef qualification_decision(*, license_verified: bool, license_ack: bool, blocking_qa: int,\n                           exact_leakage: int, acquired: bool, claim_scope_ok: bool = True) -> dict[str, str]:\n    """keep / repair / restrict / reject for one dataset, with the reason."""\n    if not acquired:\n        return dict(decision="HOLD", reason="data not acquired in this run")\n    if not license_verified:\n        return dict(decision="REJECT", reason="license not verified from official source (plan rule)")\n    if not license_ack:\n        return dict(decision="HOLD", reason="license text shown but not acknowledged by the user")\n    if exact_leakage:\n        return dict(decision="REPAIR", reason=f"{exact_leakage} exact dev/eval overlaps")\n    if blocking_qa:\n        return dict(decision="REPAIR", reason=f"{blocking_qa} blocking QA issues")\n    if not claim_scope_ok:\n        return dict(decision="RESTRICT", reason="claim scope narrower than requested")\n    return dict(decision="KEEP", reason="all gates passed for the scoped protocol")\n', 'experiments/scripts/detection_tracking_eda.py': '"""Statistics for the detection/tracking EDA notebook (compute only, no plotting).\n\nEvery function is deterministic for a fixed seed and unit-tested on cases with a\nhand-derived answer. Cluster-aware statistics are used wherever units are not\nindependent (LVVO source groups, MOT sequences).\n"""\n\nfrom __future__ import annotations\n\nimport io\nfrom collections import OrderedDict\nfrom typing import Iterable, Sequence\n\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\nfrom experiments.scripts.detection_tracking_qualification import iou_matrix_xywh\n\n\n# --------------------------------------------------------------------------\n# Dependence between units\n# --------------------------------------------------------------------------\ndef design_effect(groups: Sequence, values: Sequence[float]) -> dict[str, float]:\n    """One-way ANOVA estimate of the intra-class correlation and the design effect.\n\n    deff = 1 + (m - 1) * icc with m = sum(n_i^2) / N; n_eff = N / deff.\n    """\n    g = np.asarray(groups)\n    v = np.asarray(values, dtype=float)\n    uniq, inv, n_i = np.unique(g, return_inverse=True, return_counts=True)\n    k, n = len(uniq), len(v)\n    if k < 2 or n <= k:\n        return dict(n=n, k=k, icc=float("nan"), deff=float("nan"), n_eff=float("nan"))\n    means = np.bincount(inv, weights=v) / n_i\n    grand = v.mean()\n    msb = (n_i * (means - grand) ** 2).sum() / (k - 1)\n    msw = ((v - means[inv]) ** 2).sum() / (n - k)\n    n0 = (n - (n_i ** 2).sum() / n) / (k - 1)\n    denom = msb + (n0 - 1) * msw\n    icc = 0.0 if denom <= 0 else max(0.0, (msb - msw) / denom)\n    m = (n_i ** 2).sum() / n\n    deff = 1 + (m - 1) * icc\n    return dict(n=n, k=k, icc=float(icc), deff=float(deff), n_eff=float(n / deff))\n\n\ndef cluster_bootstrap_mean_ci(groups: Sequence, values: Sequence[float], resamples: int = 5000,\n                              seed: int = 0, level: float = 0.95) -> dict[str, float]:\n    """Percentile CI for the pooled mean, resampling whole groups (ratio estimator)."""\n    g = np.asarray(groups)\n    v = np.asarray(values, dtype=float)\n    uniq, inv = np.unique(g, return_inverse=True)\n    sums = np.bincount(inv, weights=v)\n    counts = np.bincount(inv).astype(float)\n    rng = np.random.default_rng(seed)\n    draw = rng.integers(0, len(uniq), size=(resamples, len(uniq)))\n    est = sums[draw].sum(axis=1) / counts[draw].sum(axis=1)\n    lo, hi = np.quantile(est, [(1 - level) / 2, 1 - (1 - level) / 2])\n    return dict(mean=float(v.mean()), lo=float(lo), hi=float(hi), groups=int(len(uniq)))\n\n\nclass _UnionFind:\n    def __init__(self):\n        self.parent: dict = {}\n\n    def find(self, x):\n        self.parent.setdefault(x, x)\n        while self.parent[x] != x:\n            self.parent[x] = self.parent[self.parent[x]]\n            x = self.parent[x]\n        return x\n\n    def union(self, a, b):\n        ra, rb = self.find(a), self.find(b)\n        if ra != rb:\n            self.parent[max(ra, rb)] = min(ra, rb)\n\n\ndef connected_components(nodes: Iterable, edges: Iterable[tuple]) -> dict:\n    """Map every node to a component label (smallest member). Isolated nodes are their own component."""\n    uf = _UnionFind()\n    for n in nodes:\n        uf.find(n)\n    for a, b in edges:\n        uf.union(a, b)\n    return {n: uf.find(n) for n in uf.parent}\n\n\ndef merged_group_labels(items_to_group: dict, edges: Iterable[tuple]) -> dict:\n    """Merge source groups that are linked by at least one near-duplicate edge between items."""\n    uf = _UnionFind()\n    for item, grp in items_to_group.items():\n        uf.find(grp)\n    for a, b in edges:\n        uf.union(items_to_group[a], items_to_group[b])\n    return {item: uf.find(grp) for item, grp in items_to_group.items()}\n\n\ndef ncc(a: np.ndarray, b: np.ndarray) -> float:\n    """Normalised cross-correlation of two equally-sized arrays (objective similarity, in [-1, 1])."""\n    x = np.asarray(a, dtype=float).ravel()\n    y = np.asarray(b, dtype=float).ravel()\n    x, y = x - x.mean(), y - y.mean()\n    d = np.sqrt((x * x).sum() * (y * y).sum())\n    return 0.0 if d == 0 else float((x * y).sum() / d)\n\n\ndef content_crop(gray: np.ndarray, dark: float = 12.0) -> np.ndarray:\n    """Remove letter/pillar-box bars: drop outer rows/columns that are uniformly very dark."""\n    g = np.asarray(gray, dtype=float)\n    cols = np.flatnonzero((g.mean(axis=0) > dark) | (g.std(axis=0) > 6))\n    rows = np.flatnonzero((g.mean(axis=1) > dark) | (g.std(axis=1) > 6))\n    if len(cols) == 0 or len(rows) == 0:\n        return g\n    return g[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]\n\n\ndef _resize(arr: np.ndarray, size: int) -> np.ndarray:\n    from PIL import Image\n    return np.asarray(Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8)).resize((size, size), Image.Resampling.BILINEAR), dtype=float)\n\n\ndef ink_mask(gray: np.ndarray, size: int = 128, delta: float = 40.0) -> np.ndarray:\n    """Foreground (\'ink\') mask of a slide-like frame after removing bars: |pixel - median| > delta."""\n    small = _resize(content_crop(gray), size)\n    return np.abs(small - np.median(small)) > delta\n\n\ndef slide_jaccard(a: np.ndarray, b: np.ndarray, size: int = 128, delta: float = 40.0) -> float:\n    """Jaccard overlap of the ink masks of the central slide region of two frames, in [0, 1].\n\n    Letter/pillar bars are removed and only the central 60% x 80% of the remaining frame is compared, so\n    window chrome, participant lists and webcam overlays at the edges (shared by all frames of a lecture)\n    do not inflate the score. It is a weak screening measure, not proof of identity: the notebook shows\n    calibration examples because it can still score different slides with a similar layout highly.\n    """\n    def central(g):\n        c = content_crop(g)\n        h, w = c.shape\n        return c[int(.1 * h):int(.9 * h), int(.12 * w):int(.72 * w)]\n\n    ma, mb = ink_mask(central(a), size, delta), ink_mask(central(b), size, delta)\n    union = (ma | mb).sum()\n    return float((ma & mb).sum() / union) if union else 0.0\n\n# --------------------------------------------------------------------------\n# Two-sample comparison with multiplicity control\n# --------------------------------------------------------------------------\ndef compare_samples(a: Sequence[float], b: Sequence[float]) -> dict[str, float]:\n    """KS statistic, Mann-Whitney p and Cliff\'s delta (positive: a tends to exceed b)."""\n    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)\n    ks = stats.ks_2samp(a, b)\n    mw = stats.mannwhitneyu(a, b, alternative="two-sided")\n    return dict(n_a=len(a), n_b=len(b), median_a=float(np.median(a)), median_b=float(np.median(b)),\n                ks=float(ks.statistic), p_ks=float(ks.pvalue), p_mw=float(mw.pvalue),\n                cliffs_delta=float(2 * mw.statistic / (len(a) * len(b)) - 1))\n\n\ndef holm_adjust(p: Sequence[float]) -> np.ndarray:\n    """Holm-Bonferroni adjusted p-values (monotone, capped at 1)."""\n    p = np.asarray(p, dtype=float)\n    order = np.argsort(p)\n    m = len(p)\n    adj = np.empty(m)\n    running = 0.0\n    for rank, idx in enumerate(order):\n        running = max(running, (m - rank) * p[idx])\n        adj[idx] = min(1.0, running)\n    return adj\n\n\ndef effect_label(delta: float) -> str:\n    """Romano thresholds for |Cliff\'s delta|."""\n    d = abs(delta)\n    return "negligible" if d < 0.147 else "small" if d < 0.33 else "medium" if d < 0.474 else "large"\n\n\ndef max_overlap_stats(boxes: np.ndarray) -> dict[str, np.ndarray]:\n    """For (n,4) xywh boxes of one image: per-box max IoU and max containment with any other box.\n\n    Containment = intersection / area of the smaller box (1.0 means one box lies inside the other).\n    """\n    n = len(boxes)\n    if n < 2:\n        return dict(max_iou=np.zeros(n), max_containment=np.zeros(n))\n    x1, y1 = boxes[:, 0], boxes[:, 1]\n    x2, y2 = x1 + boxes[:, 2], y1 + boxes[:, 3]\n    iw = np.clip(np.minimum(x2[:, None], x2[None]) - np.maximum(x1[:, None], x1[None]), 0, None)\n    ih = np.clip(np.minimum(y2[:, None], y2[None]) - np.maximum(y1[:, None], y1[None]), 0, None)\n    inter = iw * ih\n    area = boxes[:, 2] * boxes[:, 3]\n    smaller = np.minimum(area[:, None], area[None])\n    contain = np.divide(inter, smaller, out=np.zeros_like(inter, dtype=float), where=smaller > 0)\n    np.fill_diagonal(contain, 0.0)\n    return dict(max_iou=iou_matrix_xywh(boxes).max(axis=1), max_containment=contain.max(axis=1))\n\n\n# --------------------------------------------------------------------------\n# MOT ground-truth analytics\n# --------------------------------------------------------------------------\ndef mot_track_table(gt: np.ndarray) -> pd.DataFrame:\n    """One row per identity: appearances, span, number and length of gaps, median height."""\n    frames, ids = gt[:, 0].astype(int), gt[:, 1].astype(int)\n    rows = []\n    order = np.lexsort((frames, ids))\n    f_sorted, i_sorted, h_sorted = frames[order], ids[order], gt[order, 5]\n    bounds = np.flatnonzero(np.diff(i_sorted)) + 1\n    for f, h, tid in zip(np.split(f_sorted, bounds), np.split(h_sorted, bounds),\n                         i_sorted[np.r_[0, bounds]]):\n        gaps = np.diff(f) - 1\n        gaps = gaps[gaps > 0]\n        rows.append(dict(id=int(tid), n_frames=len(f), span=int(f[-1] - f[0] + 1), n_gaps=len(gaps),\n                         max_gap=int(gaps.max()) if len(gaps) else 0, missing=int(gaps.sum()),\n                         h_median=float(np.median(h))))\n    return pd.DataFrame(rows)\n\n\ndef mot_gap_lengths(gt: np.ndarray) -> np.ndarray:\n    """Lengths (frames) of every interruption of an identity (re-appearance after absence)."""\n    frames, ids = gt[:, 0].astype(int), gt[:, 1].astype(int)\n    order = np.lexsort((frames, ids))\n    f, i = frames[order], ids[order]\n    d = np.diff(f) - 1\n    same = i[1:] == i[:-1]\n    return d[same & (d > 0)]\n\n\ndef mot_motion(gt: np.ndarray, moving_eps: float = 0.01) -> dict[str, np.ndarray]:\n    """Per-step normalised speed (centre displacement / box height per frame) and turn angles.\n\n    Only consecutive frames are used. Turn angle (deg, 0-180) is computed for pairs of steps\n    where both speeds exceed `moving_eps`; large angles indicate non-linear motion.\n    """\n    frames, ids = gt[:, 0].astype(int), gt[:, 1].astype(int)\n    order = np.lexsort((frames, ids))\n    f, i, g = frames[order], ids[order], gt[order]\n    cx, cy, h = g[:, 2] + g[:, 4] / 2, g[:, 3] + g[:, 5] / 2, np.maximum(g[:, 5], 1e-6)\n    step = (i[1:] == i[:-1]) & (f[1:] - f[:-1] == 1)\n    dx, dy = (cx[1:] - cx[:-1])[step], (cy[1:] - cy[:-1])[step]\n    speed = np.hypot(dx, dy) / h[1:][step]\n    # angle between consecutive steps of the same track\n    idx = np.flatnonzero(step)\n    pair = (idx[1:] == idx[:-1] + 1)\n    d1x, d1y, d2x, d2y = dx[:-1][pair], dy[:-1][pair], dx[1:][pair], dy[1:][pair]\n    s1, s2 = speed[:-1][pair], speed[1:][pair]\n    ok = (s1 > moving_eps) & (s2 > moving_eps)\n    cos = (d1x * d2x + d1y * d2y)[ok] / (np.hypot(d1x, d1y)[ok] * np.hypot(d2x, d2y)[ok])\n    return dict(speed=speed, turn_deg=np.degrees(np.arccos(np.clip(cos, -1, 1))))\n\n\ndef mot_frame_overlap(gt: np.ndarray, close_iou: float = 0.5, occ_iou: float = 0.3) -> dict[str, np.ndarray]:\n    """Per-box max IoU with another GT box in the same frame, and per-frame close-pair counts."""\n    frames = gt[:, 0].astype(int)\n    boxes = gt[:, 2:6]\n    max_iou = np.zeros(len(gt))\n    uniq, start = np.unique(frames, return_index=True)\n    order = np.argsort(frames, kind="stable")\n    sorted_frames = frames[order]\n    bounds = np.r_[np.searchsorted(sorted_frames, uniq, side="left"), len(order)]\n    close_pairs = np.zeros(len(uniq), dtype=int)\n    for k in range(len(uniq)):\n        rows = order[bounds[k]:bounds[k + 1]]\n        if len(rows) < 2:\n            continue\n        iou = iou_matrix_xywh(boxes[rows])\n        max_iou[rows] = iou.max(axis=1)\n        close_pairs[k] = int((np.triu(iou, 1) > close_iou).sum())\n    return dict(max_iou=max_iou, close_pairs=close_pairs, frames=uniq,\n                occluded=max_iou > occ_iou)\n\n\ndef id_lifelines(gt: np.ndarray, occ_iou: float = 0.3) -> pd.DataFrame:\n    """Run-length segments per identity: (id, start, end, occluded).\n\n    A segment is a maximal run of consecutive frames with the same occlusion flag (box IoU > occ_iou\n    with another box of that frame). Gaps in presence split segments, so absences stay blank in a Gantt.\n    """\n    occ = mot_frame_overlap(gt, occ_iou=occ_iou)["max_iou"] > occ_iou\n    frames, ids = gt[:, 0].astype(int), gt[:, 1].astype(int)\n    order = np.lexsort((frames, ids))\n    f, i, o = frames[order], ids[order], occ[order]\n    brk = np.flatnonzero((np.diff(i) != 0) | (np.diff(f) != 1) | (o[1:] != o[:-1])) + 1\n    starts, ends = np.r_[0, brk], np.r_[brk, len(f)] - 1\n    return pd.DataFrame(dict(id=i[starts], start=f[starts], end=f[ends], occluded=o[starts]))\n\n\ndef id_confusion_pairs(gt: np.ndarray, near: float = 0.5, cross_near: float = 1.0) -> dict[str, float]:\n    """Counts that predict ID switches: how often two identities are close, and how often they swap sides.\n\n    close_pair_frames: (pair, frame) cases with centre distance < near * mean box height.\n    crossings: consecutive co-present frames where the x-order of two identities flips while the pair is\n    within cross_near * mean box height at both frames (trajectories cross, not just drift far apart).\n    """\n    frames = gt[:, 0].astype(int)\n    order = np.argsort(frames, kind="stable")\n    uniq, start = np.unique(frames[order], return_index=True)\n    bounds = np.r_[start, len(order)]\n    close = crossings = 0\n    prev: dict[tuple[int, int], tuple[int, float]] = {}\n    for k, fr in enumerate(uniq):\n        g = gt[order[bounds[k]:bounds[k + 1]]]\n        if len(g) < 2:\n            continue\n        ids = g[:, 1].astype(int)\n        cx, cy, h = g[:, 2] + g[:, 4] / 2, g[:, 3] + g[:, 5] / 2, g[:, 5]\n        hm = (h[:, None] + h[None]) / 2\n        dx = cx[:, None] - cx[None]\n        dist = np.hypot(dx, cy[:, None] - cy[None]) / np.maximum(hm, 1e-6)\n        for a, b in zip(*np.triu_indices(len(g), 1)):\n            key = (min(ids[a], ids[b]), max(ids[a], ids[b]))\n            sign = dx[a, b] if ids[a] < ids[b] else -dx[a, b]\n            near_now = dist[a, b] < cross_near\n            close += dist[a, b] < near\n            if key in prev and prev[key][0] == fr - 1 and near_now and prev[key][2] and sign * prev[key][1] < 0:\n                crossings += 1\n            prev[key] = (fr, sign, near_now)\n    return dict(close_pair_frames=int(close), crossings=int(crossings))\n\n\n# --------------------------------------------------------------------------\n# Frozen stratification and dev subset (decided on train before any tracker score exists)\n# --------------------------------------------------------------------------\ndef tertile_edges(train_values: Sequence[float]) -> tuple[float, float]:\n    """Cut points at the 1/3 and 2/3 quantiles of the train distribution."""\n    lo, hi = np.quantile(np.asarray(train_values, dtype=float), [1 / 3, 2 / 3])\n    return float(lo), float(hi)\n\n\ndef assign_tier(values: Sequence[float], edges: tuple[float, float]) -> list[str]:\n    """low: v <= e1 < mid: v <= e2 < high. Edges come from train and are applied unchanged to val."""\n    lo, hi = edges\n    return ["low" if v <= lo else "mid" if v <= hi else "high" for v in values]\n\n\ndef stratify_sequences(df: pd.DataFrame, axes: dict[str, str], moving_px: float = 1.0,\n                       cam_col: str = "cam_shift_median") -> tuple[pd.DataFrame, dict[str, tuple[float, float]]]:\n    """Add one tier column per axis (name -> feature column) plus `camera` (static/moving).\n\n    Edges use train rows only. No composite difficulty score is built.\n    """\n    train = df[df.split == "train"]\n    edges = {name: tertile_edges(train[col]) for name, col in axes.items()}\n    out = df.copy()\n    for name, col in axes.items():\n        out[f"tier_{name}"] = assign_tier(out[col], edges[name])\n    out["camera"] = np.where(out[cam_col] > moving_px, "moving", "static")\n    return out, edges\n\n\ndef select_dev_subset(strata: pd.DataFrame, n: int, seed: int, tier_cols: Sequence[str],\n                      min_per_tier: int = 2, require_moving: bool = True, max_tries: int = 20000) -> list[str]:\n    """Seeded random draw of n train sequences covering every tier of every axis >= min_per_tier times.\n\n    The first draw satisfying the constraints is returned (deterministic for a seed). Raises if none exists.\n    """\n    pool = strata[strata.split == "train"].reset_index(drop=True)\n    rng = np.random.default_rng(seed)\n    for _ in range(max_tries):\n        pick = pool.iloc[rng.choice(len(pool), size=n, replace=False)]\n        if require_moving and not (pick.camera == "moving").any():\n            continue\n        if all((pick[c].value_counts().reindex(["low", "mid", "high"]).fillna(0) >= min_per_tier).all() for c in tier_cols):\n            return sorted(pick.seq.tolist())\n    raise ValueError("no dev subset satisfies the coverage constraints; relax min_per_tier or n")\n\n\ndef coverage_vs_reference(dev: pd.DataFrame, ref: pd.DataFrame, cols: Sequence[str]) -> pd.DataFrame:\n    """Per feature: dev range, reference range, and the dev sequences outside the reference range."""\n    rows = {}\n    for c in cols:\n        lo, hi = ref[c].min(), ref[c].max()\n        out = dev[(dev[c] < lo) | (dev[c] > hi)]\n        rows[c] = dict(dev_min=dev[c].min(), dev_max=dev[c].max(), ref_min=lo, ref_max=hi,\n                       n_outside=len(out), outside=",".join(out.seq))\n    return pd.DataFrame(rows).T\n\n\n# --------------------------------------------------------------------------\n# HTTP range access to a remote ZIP (read only the members we need)\n# --------------------------------------------------------------------------\nclass HttpRangeFile(io.RawIOBase):\n    """Seekable read-only file over an HTTP server that supports Range requests."""\n\n    def __init__(self, url: str, block: int = 1 << 20, cache_blocks: int = 64, session=None):\n        import requests\n        self.url, self.block = url, block\n        self.session = session or requests.Session()\n        head = self.session.get(url, headers={"Range": "bytes=0-0"}, timeout=60)\n        head.raise_for_status()\n        self.size = int(head.headers["Content-Range"].split("/")[-1])\n        self.pos = 0\n        self._cache: OrderedDict[int, bytes] = OrderedDict()\n        self._max = cache_blocks\n\n    def seekable(self):\n        return True\n\n    def readable(self):\n        return True\n\n    def tell(self):\n        return self.pos\n\n    def seek(self, offset, whence=0):\n        self.pos = offset if whence == 0 else self.pos + offset if whence == 1 else self.size + offset\n        return self.pos\n\n    def _block(self, index: int) -> bytes:\n        if index in self._cache:\n            self._cache.move_to_end(index)\n            return self._cache[index]\n        start = index * self.block\n        end = min(start + self.block, self.size) - 1\n        last = None\n        for attempt in range(4):\n            try:\n                r = self.session.get(self.url, headers={"Range": f"bytes={start}-{end}"}, timeout=120)\n                r.raise_for_status()\n                break\n            except Exception as exc:  # retry transient network errors\n                last = exc\n        else:\n            raise ConnectionError(f"{self.url}: {last!r}")\n        self._cache[index] = r.content\n        if len(self._cache) > self._max:\n            self._cache.popitem(last=False)\n        return r.content\n\n    def read(self, n=-1):\n        if n < 0 or self.pos + n > self.size:\n            n = self.size - self.pos\n        if n <= 0:\n            return b""\n        out = bytearray()\n        while n > 0:\n            idx, off = divmod(self.pos, self.block)\n            chunk = self._block(idx)[off:off + n]\n            if not chunk:\n                break\n            out += chunk\n            self.pos += len(chunk)\n            n -= len(chunk)\n        return bytes(out)\n\n    def readinto(self, b):\n        data = self.read(len(b))\n        b[:len(data)] = data\n        return len(data)\n'}
for relative, source in SOURCES.items():
    path = WORK / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding='utf-8')
    (path.parent / '__init__.py').touch()
(WORK / 'experiments' / '__init__.py').touch()
sys.path.insert(0, str(WORK))
SOURCE_DIGESTS = {name: hashlib.sha256(src.encode()).hexdigest() for name, src in SOURCES.items()}
binding = OUT / 'source-code-identity.json'
if binding.exists() and json.loads(binding.read_text()) != SOURCE_DIGESTS:
    raise ValueError('Mã nguồn thay đổi: dùng RUN_ID mới')
binding.write_text(json.dumps(SOURCE_DIGESTS, indent=2))

import numpy as np, pandas as pd, matplotlib, requests
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from collections import Counter
from scipy import stats as sps
from experiments.scripts import detection_tracking_qualification as q
from experiments.scripts import detection_tracking_eda as eda
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': .25, 'axes.spines.top': False, 'axes.spines.right': False})
SEED = 20261026
try:
    display
except NameError:
    display = print

In [ ]:
RESULTS, OBS = {}, []     # OBS: danh sách quan sát định lượng + cờ heuristic


def savefig(fig, folder, name, show=True):
    fig.tight_layout()
    fig.savefig(folder / name, dpi=110)
    plt.close(fig)
    if show:   # backend is Agg (headless-safe), so render the saved PNG inline instead of plt.show()
        try:
            from IPython.display import display as _display, Image as _Image
            _display(_Image(filename=str(folder / name)))
        except Exception:
            print('saved', folder / name)


def save_table(df, folder, name, show=True):
    df.to_csv(folder / name)
    if show:
        display(df)


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, sort_keys=True, default=float), encoding='utf-8')


def observe(dataset, key, value, flag=None, rule=None):
    OBS.append(dict(dataset=dataset, key=key, value=value, flag=flag, rule=rule))


def download(url, dest, retries=3, timeout=60):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if OFFLINE:
        raise ConnectionError('offline test mode')
    last = None
    for attempt in range(retries):
        try:
            with requests.get(url, stream=True, timeout=timeout) as r:
                r.raise_for_status()
                tmp = dest.with_suffix(dest.suffix + '.part')
                with open(tmp, 'wb') as handle:
                    for chunk in r.iter_content(1 << 20):
                        handle.write(chunk)
            tmp.replace(dest)
            return dest
        except Exception as exc:
            last = exc
            time.sleep(2 ** attempt)
    raise ConnectionError(f'{url}: {last!r}')


def guarded(name, fn):
    try:
        RESULTS[name] = fn()
    except Exception:
        traceback.print_exc()
        RESULTS[name] = dict(error=True)
        observe(name, 'section_failed', True, flag='ERROR', rule='section raised an exception')


def load_frozen():
    reg = {k: json.loads((PRIOR / 'registry' / f).read_text(encoding='utf-8')) for k, f in (
        ('lvvo', 'lvvo-detection-protocol-261026.json'), ('coco', 'coco-subset-261026.json'),
        ('mot', 'mot-dancetrack-261026.json'))}
    gts = {}
    for key, pattern, name in (('lvvo', 'lvvo-1k-four-class', 'lvvo_gt_coco.json'), ('coco', 'coco-val2017-subset10', 'coco_subset_gt.json')):
        path = next((PRIOR / 'dataset_eda' / pattern).glob(f'*/{name}'))
        gts[key] = json.loads(path.read_text(encoding='utf-8'))
        want = reg[key]['gt_coco_canonical_sha256']
        assert q.canonical_json_sha256(gts[key]) == want, f'{key}: GT khác bản đã đóng băng'
    print('Frozen GT hashes verified (LVVO, COCO).')
    return reg, gts


REG, GT = load_frozen()

## 1. LVVO (D-A) — phân bố lớp, hình học box, cấu trúc nhóm, gần trùng

Câu hỏi: (1) lớp nào ít hỗ trợ thống kê; (2) box có đặc trưng hình học nào đáng chú ý cho detector (kích thước, tỉ lệ khung, chồng lấn, chạm biên); (3) mức phụ thuộc giữa các ảnh do nhóm nguồn và ảnh gần trùng làm giảm cỡ mẫu hiệu dụng bao nhiêu; (4) các cặp gần trùng khác nhóm thực sự giống nhau đến đâu (NCC) — có lưới ảnh để duyệt tay.

In [ ]:
LVVO_FILE_ID = '1oVufdcjhs76uDmEGu63vywKUXy5ZQSVF'
CLASS_COLORS = {'Table': '#d62728', 'Chart-Graph': '#1f77b4', 'Photographic-image': '#2ca02c', 'Visual-illustration': '#ff7f0e'}


JAC_THRESHOLD = 0.5   # screening cut only; the calibration figure shows how reliable (or not) it is


def thumb(path, width=480):
    with Image.open(path) as im:
        g = im.convert('L')
        return np.asarray(g.resize((width, max(1, round(g.height * width / g.width))), Image.Resampling.BILINEAR), dtype=np.uint8)


def eda_lvvo():
    folder = OUT / 'lvvo'
    folder.mkdir(exist_ok=True)
    archive = WORK / 'lvvo' / 'LVVO_1k_withCategories.zip'
    archive.parent.mkdir(parents=True, exist_ok=True)
    if os.environ.get('DT_LVVO_ZIP') and not archive.exists():
        shutil.copyfile(os.environ['DT_LVVO_ZIP'], archive)
    if not archive.exists():
        import gdown
        if OFFLINE or not gdown.download(id=LVVO_FILE_ID, output=str(archive), quiet=True):
            raise ConnectionError('Không tải được LVVO')
    assert q.sha256_file(archive) == REG['lvvo']['archive_sha256'], 'archive khác bản đã đóng băng'
    q.safe_extract(archive, WORK / 'lvvo' / 'source')
    root = WORK / 'lvvo' / 'source' / 'LVVO_1k_withCategories'

    gt = GT['lvvo']
    names = {c['id']: c['name'] for c in gt['categories']}
    ledger = pd.read_csv(next((PRIOR / 'dataset_eda' / 'lvvo-1k-four-class').glob('*/image_ledger.csv'))).set_index('image_id')
    img = pd.DataFrame(gt['images']).set_index('id').join(ledger[['stem', 'group']])
    ann = pd.DataFrame(gt['annotations'])
    ann['cls'] = ann.category_id.map(names)
    ann = ann.join(img[['width', 'height', 'group', 'stem']], on='image_id')
    ann[['x', 'y', 'w', 'h']] = pd.DataFrame(ann.bbox.tolist(), index=ann.index)
    ann['area_frac'] = ann.area / (ann.width * ann.height)
    ann['log2_aspect'] = np.log2(ann.w / ann.h)
    ann['cx'], ann['cy'] = (ann.x + ann.w / 2) / ann.width, (ann.y + ann.h / 2) / ann.height
    ann['touch'] = (ann.x <= 1) | (ann.y <= 1) | (ann.x + ann.w >= ann.width - 1) | (ann.y + ann.h >= ann.height - 1)
    ov = {i: eda.max_overlap_stats(g[['x', 'y', 'w', 'h']].to_numpy(float)) for i, g in ann.groupby('image_id')}
    ann['max_iou'] = 0.0
    ann['max_contain'] = 0.0
    for i, g in ann.groupby('image_id'):
        ann.loc[g.index, 'max_iou'] = ov[i]['max_iou']
        ann.loc[g.index, 'max_contain'] = ov[i]['max_containment']
    classes = sorted(names.values())

    # ---- 1.1 per-class summary (descriptive) ----
    per = ann.groupby('cls').agg(boxes=('id', 'size'), images=('image_id', 'nunique'),
                                 area_frac_p05=('area_frac', lambda s: s.quantile(.05)), area_frac_med=('area_frac', 'median'),
                                 area_frac_p95=('area_frac', lambda s: s.quantile(.95)),
                                 log2_aspect_med=('log2_aspect', 'median'), touch_border=('touch', 'mean'),
                                 overlap_iou_gt_0_3=('max_iou', lambda s: (s > .3).mean()),
                                 nested_contain_gt_0_9=('max_contain', lambda s: (s > .9).mean()))
    per['box_share'] = per.boxes / per.boxes.sum()
    per['boxes_per_positive_image'] = per.boxes / per.images
    cis = {}
    n_img = len(img)
    for c in classes:
        counts = ann[ann.cls == c].groupby('image_id').size().reindex(img.index, fill_value=0)
        cis[c] = eda.cluster_bootstrap_mean_ci(img.group.values, counts.values, seed=SEED)
    per['mean_boxes_per_image'] = [cis[c]['mean'] for c in per.index]
    per['mpi_ci_lo'] = [cis[c]['lo'] for c in per.index]
    per['mpi_ci_hi'] = [cis[c]['hi'] for c in per.index]
    save_table(per.round(4), folder, 'per_class_summary.csv')
    for c in classes:
        sup = int(per.loc[c, 'boxes'])
        observe('lvvo', f'boxes[{c}]', sup, flag='LOW_SUPPORT' if sup < 150 else None, rule='boxes < 150 (heuristic: per-class AP unstable)')

    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    per.boxes.sort_values().plot.barh(ax=ax[0], color=[CLASS_COLORS[c] for c in per.boxes.sort_values().index], title='Boxes per class')
    for c in classes:
        sub = ann[ann.cls == c]
        ax[1].hist(np.log10(sub.area_frac), bins=30, histtype='step', lw=2, density=True, label=c, color=CLASS_COLORS[c])
        ax[2].hist(sub.log2_aspect, bins=30, histtype='step', lw=2, density=True, label=c, color=CLASS_COLORS[c])
    ax[1].set(title='log10(box area / image area)', xlabel='log10 fraction')
    ax[2].set(title='log2(width/height): 0 = square', xlabel='log2 aspect')
    ax[1].legend(fontsize=7)
    savefig(fig, folder, 'class_geometry.png')

    # ---- 1.2 spatial prior, co-occurrence ----
    fig, ax = plt.subplots(1, 4, figsize=(16, 3.6))
    for a, c in zip(ax, classes):
        sub = ann[ann.cls == c]
        h, _, _ = np.histogram2d(sub.cy, sub.cx, bins=12, range=[[0, 1], [0, 1]])
        a.imshow(h / h.sum(), cmap='magma', origin='upper')
        a.set_title(f'{c}\ncentre density', fontsize=9)
        a.grid(False)
    savefig(fig, folder, 'center_density.png')
    pres = pd.DataFrame({c: img.index.isin(ann[ann.cls == c].image_id) for c in classes}, index=img.index)
    co = pd.DataFrame([[ (pres[a] & pres[b]).sum() / max((pres[a] | pres[b]).sum(), 1) for b in classes] for a in classes], index=classes, columns=classes)
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    im = ax.imshow(co.values, vmin=0, vmax=1, cmap='viridis')
    ax.set_xticks(range(4), classes, rotation=30, ha='right')
    ax.set_yticks(range(4), classes)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{co.values[i, j]:.2f}', ha='center', va='center', color='w')
    ax.set_title('Image-level co-occurrence (Jaccard)')
    ax.grid(False)
    fig.colorbar(im)
    savefig(fig, folder, 'cooccurrence.png')
    save_table(co.round(3), folder, 'cooccurrence_jaccard.csv', show=False)

    # ---- 1.3 groups, dependence, image size ----
    grp = img.group.value_counts()
    obs_groups = dict(groups=len(grp), median=float(grp.median()), max=int(grp.max()), top5_share=float(grp.head(5).sum() / n_img))
    print('groups:', obs_groups)
    observe('lvvo', 'groups', obs_groups)
    size = img.groupby(['width', 'height']).size().sort_values(ascending=False)
    print('top image sizes:', size.head(6).to_dict())
    mix = ann.pivot_table(index='group', columns='cls', values='id', aggfunc='size', fill_value=0).reindex(grp.index).fillna(0)
    top = mix.head(40)
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    grp.plot.hist(bins=30, ax=ax[0], title=f'Images per source group (n groups={len(grp)})')
    ax[1].imshow((top.div(top.sum(axis=1), axis=0)).values, aspect='auto', cmap='viridis')
    ax[1].set_xticks(range(4), top.columns, rotation=30, ha='right')
    ax[1].set_title('Class mix of the 40 largest groups (row-normalised)')
    ax[1].grid(False)
    savefig(fig, folder, 'groups.png')

    dep_rows = []
    for c in classes:
        v = pres[c].astype(float).values
        d = eda.design_effect(img.group.values, v)
        dep_rows.append(dict(outcome=f'present[{c}]', **d))
    d = eda.design_effect(img.group.values, img.index.map(ann.groupby('image_id').size()).fillna(0).values)
    dep_rows.append(dict(outcome='boxes per image', **d))
    dep = pd.DataFrame(dep_rows).set_index('outcome')

    # ---- 1.4 near-duplicates: objective similarity + clusters ----
    pairs = pd.read_csv(next((PRIOR / 'dataset_eda' / 'lvvo-1k-four-class').glob('*/near_duplicate_pairs_dhash4.csv')))
    cache = {}

    def th(stem):
        if stem not in cache:
            cache[stem] = thumb(root / 'images' / f'{stem}.jpg')
        return cache[stem]

    # Raw-pixel NCC is kept only to show why it is unsuitable here: slides share a white page and black bars.
    pairs['ncc'] = [eda.ncc(eda._resize(eda.content_crop(th(a).astype(float)), 64), eda._resize(eda.content_crop(th(b).astype(float)), 64))
                    for a, b in zip(pairs.left, pairs.right)]
    pairs['jac'] = [eda.slide_jaccard(th(a), th(b)) for a, b in zip(pairs.left, pairs.right)]
    pairs['likely_duplicate'] = pairs.jac >= JAC_THRESHOLD
    pairs.sort_values('jac', ascending=False).to_csv(folder / 'near_duplicate_pairs_with_similarity.csv', index=False)
    fig, ax = plt.subplots(1, 3, figsize=(17, 3.8))
    for flag, g in pairs.groupby('same_group'):
        ax[0].hist(g.jac, bins=25, alpha=.6, label=f'same group={flag} (n={len(g)})')
    ax[0].axvline(JAC_THRESHOLD, color='k', ls='--')
    ax[0].set(title='Slide Jaccard of dHash<=4 pairs (central region)', xlabel='slide Jaccard')
    ax[0].legend()
    ax[1].scatter(pairs.jac, pairs.ncc, s=12, c=pairs.same_group.map({True: '#1f77b4', False: '#d62728'}))
    ax[1].set(xlabel='slide Jaccard', ylabel='raw NCC', title='NCC is high even for different slides (red = cross-group)')
    ax[2].scatter(pairs.distance, pairs.jac, s=12, c=pairs.same_group.map({True: '#1f77b4', False: '#d62728'}))
    ax[2].set(xlabel='dHash Hamming distance', ylabel='slide Jaccard', title='dHash vs slide Jaccard')
    savefig(fig, folder, 'near_duplicate_similarity.png')

    # calibration figure: examples from three similarity bands, so the threshold can be judged by eye
    rng_cal = np.random.default_rng(SEED)
    bands = [('jac >= 0.3', pairs[pairs.jac >= .3]), ('0.15 <= jac < 0.3', pairs[(pairs.jac >= .15) & (pairs.jac < .3)]), ('jac < 0.15', pairs[pairs.jac < .15])]
    fig, axes = plt.subplots(3, 8, figsize=(20, 7.5))
    for r, (label, band) in enumerate(bands):
        pick = band.iloc[rng_cal.permutation(len(band))[:4]] if len(band) else band
        for k in range(4):
            for side in range(2):
                a = axes[r, k * 2 + side]
                a.axis('off')
                if k < len(pick):
                    row = pick.iloc[k]
                    stem = (row.left, row.right)[side]
                    with Image.open(root / 'images' / f'{stem}.jpg') as im:
                        a.imshow(im.convert('RGB').resize((320, 180)))
                    a.set_title(f'{label} | jac={row.jac:.2f} ncc={row.ncc:.2f}' if side == 0 else f'{"same" if row.same_group else "cross"}-group', fontsize=7)
    fig.suptitle('Calibration: random dHash<=4 pairs per slide-Jaccard band (left | right of each pair): judge by eye whether high bands are really the same slide')
    savefig(fig, folder, 'near_duplicate_calibration.png')
    stem_group = dict(zip(img.stem, img.group))
    edges_all = list(zip(pairs.left, pairs.right))
    edges_hi = list(zip(pairs[pairs.likely_duplicate].left, pairs[pairs.likely_duplicate].right))
    comp_all = eda.connected_components(img.stem, edges_all)
    comp_hi = eda.connected_components(img.stem, edges_hi)
    sizes_all = pd.Series(comp_all).value_counts()
    sizes_hi = pd.Series(comp_hi).value_counts()
    summary_dup = dict(pairs=len(pairs), cross_group=int((~pairs.same_group).sum()), likely_duplicate=int(pairs.likely_duplicate.sum()),
                       likely_duplicate_cross_group=int((pairs.likely_duplicate & ~pairs.same_group).sum()),
                       images_in_any_pair=int((sizes_all[sizes_all > 1]).sum()), largest_component=int(sizes_all.max()),
                       images_in_likely_duplicate_pairs=int(sizes_hi[sizes_hi > 1].sum()))
    print('near-duplicate summary:', summary_dup)
    observe('lvvo', 'near_duplicates', summary_dup, flag='REVIEW' if summary_dup['likely_duplicate_cross_group'] else None,
            rule='cross-group pairs with slide Jaccard >= 0.5: screening flag only, inspect contact sheets')
    merged = eda.merged_group_labels(stem_group, edges_all)
    merged_hi = eda.merged_group_labels(stem_group, edges_hi)
    img['merged_group'] = img.stem.map(merged)
    img['merged_group_hi'] = img.stem.map(merged_hi)
    for label, col in (('merged by all dHash pairs (upper bound on dependence)', 'merged_group'), ('merged by slide Jaccard>=0.5 pairs', 'merged_group_hi')):
        for c in classes:
            dep_rows.append(dict(outcome=f'present[{c}] | {label}', **eda.design_effect(img[col].values, pres[c].astype(float).values)))
    dep = pd.DataFrame(dep_rows).set_index('outcome')
    save_table(dep.round(3), folder, 'design_effect.csv')
    cl = {name: img[col].nunique() for name, col in (('author groups', 'group'), ('merged (all dHash pairs)', 'merged_group'), ('merged (jac>=0.5)', 'merged_group_hi'))}
    print('independent clusters available for the bootstrap:', cl)
    observe('lvvo', 'independent_clusters', cl)
    own = dep[dep.index.str.startswith('present') & ~dep.index.str.contains(r'\|')]
    upper = dep[dep.index.str.contains('all dHash pairs')]
    observe('lvvo', 'max_design_effect_author_groups', float(own.deff.max()), flag='DEPENDENT' if own.deff.max() > 1.5 else None,
            rule='design effect > 1.5 with author groups: bootstrap must resample groups')
    observe('lvvo', 'max_design_effect_upper_bound_all_dhash_merge', float(upper.deff.max()),
            rule='upper bound if every dHash<=4 pair were a true duplicate (not supported by calibration figure)')
    observe('lvvo', 'min_effective_n_author_groups', float(own.n_eff.min()))

    # contact sheets for cross-group pairs (manual adjudication aid)
    cross = pairs[~pairs.same_group].sort_values('jac', ascending=False).reset_index(drop=True)
    per_page, pages = 12, int(np.ceil(len(cross) / 12))
    sheet_dir = folder / 'dup_review_sheets'
    sheet_dir.mkdir(exist_ok=True)
    for p in range(pages):
        chunk = cross.iloc[p * per_page:(p + 1) * per_page]
        fig, axes = plt.subplots(6, 4, figsize=(14, 17))
        for a in axes.ravel():
            a.axis('off')
        for k, r in enumerate(chunk.itertuples()):
            for side, stem in enumerate((r.left, r.right)):
                a = axes[k // 2, (k % 2) * 2 + side]
                with Image.open(root / 'images' / f'{stem}.jpg') as im:
                    a.imshow(im.convert('RGB').resize((320, 180)))
                a.set_title(f'{stem[:34]}\n{"jac %.2f ncc %.2f d=%d" % (r.jac, r.ncc, r.distance) if side == 0 else "group " + stem_group[stem][:20]}', fontsize=6)
        fig.suptitle(f'Cross-group dHash<=4 pairs, sorted by slide Jaccard (page {p + 1}/{pages}); ghi nhận xét vào cột reviewer của CSV', fontsize=9)
        savefig(fig, sheet_dir, f'page_{p + 1:02d}.png', show=p < 2)
    template = cross[['left', 'right', 'distance', 'jac', 'ncc']].assign(reviewer_decision='', note='')
    template.to_csv(folder / 'cross_group_pairs_review_template.csv', index=False)

    # ---- 1.5 overlays ----
    rng = np.random.default_rng(SEED)
    fig, axes = plt.subplots(4, 4, figsize=(16, 13))
    for r, c in enumerate(classes):
        ids = rng.choice(ann[ann.cls == c].image_id.unique(), 4, replace=False)
        for k, i in enumerate(ids):
            a = axes[r, k]
            im = Image.open(root / 'images' / (img.loc[i, 'stem'] + '.jpg')).convert('RGB')
            d = ImageDraw.Draw(im)
            for _, b in ann[ann.image_id == i].iterrows():
                d.rectangle((b.x, b.y, b.x + b.w, b.y + b.h), outline=CLASS_COLORS[b.cls], width=max(3, im.width // 300))
            a.imshow(im)
            a.set_title(f'{c} | {img.loc[i, "stem"][:30]}', fontsize=7)
            a.axis('off')
    fig.suptitle('GT overlays: one row per class (colour = class of each box)')
    savefig(fig, folder, 'overlays.png')
    return dict(per_class=per.boxes.to_dict(), images=n_img, groups=len(grp))


guarded('lvvo', eda_lvvo)

## 2. COCO subset (D-B) — hỗ trợ theo lớp, tính đại diện, hình học, chồng lấn

Câu hỏi: (1) hạn ngạch 200 ảnh/lớp có đạt không và lớp nào bị giới hạn bởi dữ liệu; (2) tập con lệch gì so với toàn bộ val2017 (lệch do phân tầng là có chủ đích, cần biết hướng và độ lớn); (3) hình học và mức chồng lấn/chạm biên theo lớp; (4) độ chính xác ước lượng được theo lớp (bootstrap theo ảnh).

In [ ]:
COCO_ANN_URL = 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip'
COCO_IMG_URL = 'http://images.cocodataset.org/val2017/{}'


def eda_coco():
    from concurrent.futures import ThreadPoolExecutor
    folder = OUT / 'coco'
    folder.mkdir(exist_ok=True)
    cdir = WORK / 'coco'
    instances = cdir / 'annotations' / 'instances_val2017.json'
    if not instances.exists():
        zpath = cdir / 'annotations_trainval2017.zip'
        if not zpath.exists():
            download(COCO_ANN_URL, zpath)
        assert q.sha256_file(zpath) == REG['coco']['annotations_zip_sha256'], 'annotations khác bản đã đóng băng'
        q.safe_extract(zpath, cdir, keep=lambda n: n.endswith('instances_val2017.json'))
    full = json.loads(instances.read_text(encoding='utf-8'))
    tax = {int(k): v for k, v in REG['coco']['taxonomy'].items()}
    classes = list(tax.values())
    subset = GT['coco']
    ids = set(REG['coco']['image_ids'])
    img = pd.DataFrame(subset['images']).set_index('id')
    ann = pd.DataFrame(subset['annotations'])
    ann['cls'] = ann.category_id.map(tax)
    ann = ann.join(img[['width', 'height', 'file_name', 'license']], on='image_id')
    ann[['x', 'y', 'w', 'h']] = pd.DataFrame(ann.bbox.tolist(), index=ann.index)
    ann['area_frac'] = ann.area / (ann.width * ann.height)
    ann['log2_aspect'] = np.log2(ann.w.clip(lower=1) / ann.h.clip(lower=1))
    ann['stratum'] = ann.area.map(q.size_stratum)
    ann['cx'], ann['cy'] = (ann.x + ann.w / 2) / ann.width, (ann.y + ann.h / 2) / ann.height
    ann['touch'] = (ann.x <= 1) | (ann.y <= 1) | (ann.x + ann.w >= ann.width - 1) | (ann.y + ann.h >= ann.height - 1)
    ann['max_iou'] = 0.0
    ann['max_contain'] = 0.0
    for i, g in ann[ann.iscrowd == 0].groupby('image_id'):
        o = eda.max_overlap_stats(g[['x', 'y', 'w', 'h']].to_numpy(float))
        ann.loc[g.index, 'max_iou'] = o['max_iou']
        ann.loc[g.index, 'max_contain'] = o['max_containment']
    nc = ann[ann.iscrowd == 0]

    # ---- 2.1 quota vs availability ----
    fa = pd.DataFrame(full['annotations'])
    fa = fa[(fa.category_id.isin(tax)) & (fa.iscrowd == 0)]
    fa['cls'] = fa.category_id.map(tax)
    avail = fa.groupby('cls').image_id.nunique()
    got = nc.groupby('cls').image_id.nunique()
    quota = REG['coco']['selection']['quota_per_class']
    sup = pd.DataFrame({'eligible_images_full_val': avail, 'selected_images': got, 'quota': quota,
                        'selected_boxes': nc.groupby('cls').size(), 'full_val_boxes': fa.groupby('cls').size()}).loc[classes]
    sup['quota_met'] = sup.selected_images >= sup.quota
    sup['selected_share_of_available'] = (sup.selected_images / sup.eligible_images_full_val).round(3)
    save_table(sup, folder, 'support_vs_quota.csv')
    for c in classes:
        r = sup.loc[c]
        observe('coco', f'selected_images[{c}]', int(r.selected_images), flag=None if r.quota_met else 'QUOTA_NOT_MET',
                rule=f'selected images < quota {quota} (class limited by availability or by overlap with other classes)')
        observe('coco', f'boxes[{c}]', int(r.selected_boxes), flag='LOW_SUPPORT' if r.selected_boxes < 150 else None,
                rule='boxes < 150 (heuristic: per-class AP unstable)')
    fig, ax = plt.subplots(figsize=(11, 4))
    x = np.arange(len(classes))
    ax.bar(x - .2, sup.eligible_images_full_val, .4, label='images with class in full val2017', color='#bbb')
    ax.bar(x + .2, sup.selected_images, .4, label='selected images', color='#1f77b4')
    ax.axhline(quota, color='k', ls='--', label=f'quota {quota}')
    ax.set_xticks(x, classes, rotation=30, ha='right')
    ax.set_yscale('log')
    ax.legend()
    ax.set_title('Per-class availability vs selection')
    savefig(fig, folder, 'quota_vs_availability.png')

    # ---- 2.2 representativeness vs full val (descriptive; selection is stratified on purpose) ----
    rows = []
    fa['area'] = fa.area
    for c in classes:
        a_full = np.log2(fa[fa.cls == c].area.clip(lower=1))
        a_sub = np.log2(nc[nc.cls == c].area.clip(lower=1))
        r = eda.compare_samples(a_sub, a_full)
        rows.append(dict(cls=c, n_subset=len(a_sub), n_full=len(a_full), median_log2_area_subset=r['median_a'],
                         median_log2_area_full=r['median_b'], ks=r['ks'], cliffs_delta=r['cliffs_delta'],
                         small_share_subset=(nc[nc.cls == c].stratum == 'small').mean(),
                         small_share_full=(fa[fa.cls == c].area < 32 ** 2).mean()))
    rep = pd.DataFrame(rows).set_index('cls')
    rep['effect'] = rep.cliffs_delta.map(eda.effect_label)
    save_table(rep.round(3), folder, 'representativeness_vs_full_val.csv')
    for c in classes:
        observe('coco', f'size_shift_vs_full[{c}]', dict(delta=float(rep.loc[c, 'cliffs_delta']), effect=rep.loc[c, 'effect']),
                flag='SIZE_SHIFT' if abs(rep.loc[c, 'cliffs_delta']) >= .33 else None,
                rule='|Cliff delta| >= 0.33 between subset and full val2017 box areas (expected under stratification; report direction)')
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    w = .4
    ax[0].bar(x - w / 2, rep.small_share_full, w, label='full val2017', color='#bbb')
    ax[0].bar(x + w / 2, rep.small_share_subset, w, label='subset', color='#1f77b4')
    ax[0].set_xticks(x, classes, rotation=30, ha='right')
    ax[0].set(title='Share of small boxes (<32^2 px)')
    ax[0].legend()
    strata = nc.pivot_table(index='cls', columns='stratum', values='id', aggfunc='size', fill_value=0).loc[classes][['small', 'medium', 'large']]
    (strata.div(strata.sum(axis=1), axis=0)).plot.bar(stacked=True, ax=ax[1], title='Box size strata in subset (share)', color=['#d62728', '#ff7f0e', '#2ca02c'])
    ax[1].tick_params(axis='x', rotation=30)
    savefig(fig, folder, 'representativeness_size.png')

    # ---- 2.3 geometry, spatial prior, crowd, overlap, truncation ----
    per = nc.groupby('cls').agg(boxes=('id', 'size'), images=('image_id', 'nunique'), area_frac_med=('area_frac', 'median'),
                                log2_aspect_med=('log2_aspect', 'median'), touch_border=('touch', 'mean'),
                                overlap_iou_gt_0_3=('max_iou', lambda s: (s > .3).mean()),
                                nested_contain_gt_0_9=('max_contain', lambda s: (s > .9).mean())).loc[classes]
    per['crowd_boxes'] = ann[ann.iscrowd == 1].groupby('cls').size().reindex(classes).fillna(0).astype(int)
    per['boxes_per_positive_image'] = per.boxes / per.images
    ci = {c: eda.cluster_bootstrap_mean_ci(nc[nc.cls == c].image_id.values, (nc[nc.cls == c].stratum == 'small').astype(float).values, seed=SEED) for c in classes}
    per['small_share'] = [ci[c]['mean'] for c in classes]
    per['small_ci_lo'] = [ci[c]['lo'] for c in classes]
    per['small_ci_hi'] = [ci[c]['hi'] for c in classes]
    save_table(per.round(4), folder, 'per_class_summary.csv')
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    for c in classes:
        s = nc[nc.cls == c]
        ax[0].hist(np.log10(s.area_frac), bins=30, histtype='step', density=True, label=c)
        ax[1].hist(s.log2_aspect, bins=30, histtype='step', density=True)
    ax[0].set(title='log10(area / image area)')
    ax[0].legend(fontsize=6, ncol=2)
    ax[1].set(title='log2(w/h)')
    nc.groupby('image_id').size().plot.hist(bins=30, ax=ax[2], title='Taxonomy boxes per image')
    savefig(fig, folder, 'geometry.png')
    fig, ax = plt.subplots(2, 5, figsize=(16, 6.5))
    for a, c in zip(ax.ravel(), classes):
        s = nc[nc.cls == c]
        h, _, _ = np.histogram2d(s.cy, s.cx, bins=12, range=[[0, 1], [0, 1]])
        a.imshow(h / h.sum(), cmap='magma')
        a.set_title(c, fontsize=9)
        a.grid(False)
        a.axis('off')
    fig.suptitle('Box-centre density (normalised image coordinates)')
    savefig(fig, folder, 'center_density.png')
    pres = pd.DataFrame({c: img.index.isin(nc[nc.cls == c].image_id) for c in classes}, index=img.index)
    co = pd.DataFrame([[(pres[a] & pres[b]).sum() / max((pres[a] | pres[b]).sum(), 1) for b in classes] for a in classes], index=classes, columns=classes)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(co.values, cmap='viridis')
    ax.set_xticks(range(len(classes)), classes, rotation=45, ha='right')
    ax.set_yticks(range(len(classes)), classes)
    ax.set_title('Image-level co-occurrence (Jaccard)')
    ax.grid(False)
    fig.colorbar(im)
    savefig(fig, folder, 'cooccurrence.png')
    save_table(co.round(3), folder, 'cooccurrence_jaccard.csv', show=False)

    # ---- 2.4 images, licences ----
    lic = {l['id']: l['name'] for l in full.get('licenses', [])}
    lic_counts = img.license.map(lic).value_counts()
    restricted = img.license.map(lic).str.contains('NonCommercial|NoDerivs', na=False).mean()
    save_table(lic_counts.to_frame('images'), folder, 'image_licenses.csv')
    observe('coco', 'share_images_NonCommercial_or_NoDerivs', float(restricted), flag='LICENCE', rule='research use only, no redistribution of images')
    sizes = img.groupby(['width', 'height']).size().sort_values(ascending=False)
    print('top image sizes:', sizes.head(5).to_dict(), '| distinct sizes:', len(sizes))

    # ---- 2.5 overlays (downloads only the sampled images) ----
    rng = np.random.default_rng(SEED)
    fig, axes = plt.subplots(5, 6, figsize=(18, 14))
    imgdir = cdir / 'val2017_subset'
    for r, c in enumerate(classes):
        pool = nc[nc.cls == c].image_id.unique()
        pick = rng.choice(pool, 3, replace=False)
        for k, i in enumerate(pick):
            a = axes[r // 2, (r % 2) * 3 + k]
            fn = img.loc[i, 'file_name']
            path = imgdir / fn
            if not path.exists():
                download(COCO_IMG_URL.format(fn), path)
            im = Image.open(path).convert('RGB')
            d = ImageDraw.Draw(im)
            for _, b in ann[ann.image_id == i].iterrows():
                col = '#ff3b30' if b.cls == c else '#00c2ff'
                d.rectangle((b.x, b.y, b.x + b.w, b.y + b.h), outline=col, width=3)
            a.imshow(im)
            a.set_title(f'{c} | {fn}', fontsize=7)
            a.axis('off')
    fig.suptitle('Overlays (red = focus class, cyan = other taxonomy classes)')
    savefig(fig, folder, 'overlays.png')
    return dict(classes=classes)


guarded('coco', eda_coco)

## 3. DanceTrack (T) — độ khó association, dịch chuyển train/val, camera

Mỗi sequence là một đơn vị. Đặc trưng độ khó liên quan trực tiếp đến AssA/HOTA: mật độ, **che khuất** (IoU>0.3 với box khác), **cặp gần nhau** (IoU>0.5, nguy cơ đổi ID), **khoảng gián đoạn của ID** (tái xuất hiện), **vận tốc chuẩn hóa** theo chiều cao box và **tính phi tuyến** của chuyển động (góc rẽ giữa hai bước liên tiếp). So sánh train (tuning) với val (đánh giá) ở mức sequence, Holm cho nhiều đặc trưng. Camera: phase correlation trên các cặp frame liên tiếp, dùng để kiểm chứng proxy cũ, không phải nhãn.

In [ ]:
DT_BASE = 'https://huggingface.co/datasets/noahcao/dancetrack/resolve/main/{}.zip'
CAM_PAIRS = 6
DT = {}   # state shared with section 3b (sequence table, GT, open zip members)
FEATURES = [('boxes_per_frame', 'mật độ box/frame'), ('occluded_share', 'tỉ lệ box bị che (IoU>0.3)'),
            ('close_pairs_per_1k_frames', 'cặp IoU>0.5 / 1000 frame'), ('gap_missing_share', 'tỉ lệ frame vắng của ID'),
            ('gaps_per_track', 'số gián đoạn / track'), ('track_len_median', 'độ dài track trung vị'),
            ('h_rel_median', 'chiều cao box / chiều cao ảnh'), ('speed_median', 'vận tốc chuẩn hóa trung vị'),
            ('speed_p90', 'vận tốc chuẩn hóa p90'), ('turn_deg_mean', 'góc rẽ trung bình (độ)'),
            ('cam_shift_median', 'dịch chuyển camera trung vị (px)')]


def eda_dancetrack():
    import cv2
    folder = OUT / 'dancetrack'
    folder.mkdir(exist_ok=True)
    zips = {}
    for name in ('train1', 'train2', 'val'):
        zips[name] = zipfile.ZipFile(eda.HttpRangeFile(DT_BASE.format(name)))
    seqs = {}
    for name, zf in zips.items():
        default = 'val' if name == 'val' else 'train'
        for n in (i.filename for i in zf.infolist() if not i.is_dir()):
            m = q.parse_mot_member(n, default)
            if not m:
                continue
            e = seqs.setdefault((m['split'], m['seq']), dict(zf=zf, frames={}, gt=None, info=None))
            if m['kind'] == 'seqinfo.ini':
                e['info'] = n
            elif m['kind'] == 'gt/gt.txt':
                e['gt'] = n
            else:
                e['frames'][int(m['frame'])] = n
    frozen = {(s, d['seq']): d for s in ('train', 'val') for d in REG['mot']['sequences'][s]}
    assert set(frozen) <= set(seqs), 'sequence đã đóng băng không còn trong nguồn'
    rows, pooled = [], {'train': dict(track_len=[], gap=[], speed=[], turn=[], h_rel=[], bpf=[]), 'val': dict(track_len=[], gap=[], speed=[], turn=[], h_rel=[], bpf=[])}
    gts, timeline = {}, {}
    for (split, seq), d in sorted(frozen.items()):
        e = seqs[(split, seq)]
        gt_bytes = e['zf'].read(e['gt'])
        assert hashlib.sha256(gt_bytes).hexdigest() == d['gt_sha256'], f'{seq}: GT khác bản đã đóng băng'
        info = q.parse_seqinfo(e['zf'].read(e['info']).decode())
        length, W, H = int(info['seqLength']), int(info['imWidth']), int(info['imHeight'])
        gt = q.parse_mot_gt(gt_bytes.decode())
        gts[(split, seq)] = gt
        tracks = eda.mot_track_table(gt)
        gaps = eda.mot_gap_lengths(gt)
        mo = eda.mot_motion(gt)
        ov = eda.mot_frame_overlap(gt)
        per_frame = np.bincount(gt[:, 0].astype(int), minlength=length + 1)[1:]
        timeline[(split, seq)] = per_frame
        # camera: consecutive pairs spread over the video
        starts = np.linspace(1, max(length - 1, 1), CAM_PAIRS).astype(int)
        pairs_img = []
        for s0 in starts:
            if s0 in e['frames'] and s0 + 1 in e['frames']:
                ims = []
                for fidx in (s0, s0 + 1):
                    arr = cv2.imdecode(np.frombuffer(e['zf'].read(e['frames'][fidx]), np.uint8), cv2.IMREAD_GRAYSCALE)
                    ims.append(cv2.resize(arr, (480, max(1, int(arr.shape[0] * 480 / arr.shape[1])))))
                pairs_img.append(tuple(ims))
        shifts = [float(np.hypot(*cv2.phaseCorrelate(np.float32(a), np.float32(b))[0])) for a, b in pairs_img]
        rows.append(dict(
            split=split, seq=seq, frames=length, width=W, height=H, tracks=len(tracks), boxes=len(gt),
            boxes_per_frame=len(gt) / length, occluded_share=float(ov['occluded'].mean()),
            close_pairs_per_1k_frames=float(ov['close_pairs'].sum() / length * 1000),
            gap_missing_share=float(tracks.missing.sum() / tracks.span.sum()), gaps_per_track=float(tracks.n_gaps.mean()),
            track_len_median=float(tracks.n_frames.median()), h_rel_median=float(np.median(gt[:, 5]) / H),
            speed_median=float(np.median(mo['speed'])), speed_p90=float(np.quantile(mo['speed'], .9)),
            turn_deg_mean=float(mo['turn_deg'].mean()) if len(mo['turn_deg']) else np.nan,
            cam_shift_median=float(np.median(shifts)) if shifts else np.nan, cam_pairs=len(shifts),
            ids_min=int(gt[:, 1].min()), ids_max=int(gt[:, 1].max()), ids_contiguous=bool(len(np.unique(gt[:, 1])) == gt[:, 1].max() - gt[:, 1].min() + 1)))
        p = pooled[split]
        p['track_len'] += tracks.n_frames.tolist()
        p['gap'] += gaps.tolist()
        p['speed'] += mo['speed'].tolist()
        p['turn'] += mo['turn_deg'].tolist()
        p['h_rel'] += (gt[:, 5] / H).tolist()
        p['bpf'] += per_frame.tolist()
    df = pd.DataFrame(rows)
    save_table(df.round(4), folder, 'sequence_features.csv', show=False)
    display(df.groupby('split')[[f for f, _ in FEATURES]].median().round(3))
    print('ID bắt đầu từ 0 và liên tục trong mọi sequence:', bool((df.ids_min == 0).all() and df.ids_contiguous.all()))

    # ---- 3.1 train vs val (sequence-level inference, Holm) ----
    tr, va = df[df.split == 'train'], df[df.split == 'val']
    comp = pd.DataFrame({f: eda.compare_samples(tr[f].dropna(), va[f].dropna()) for f, _ in FEATURES}).T
    comp['p_mw_holm'] = eda.holm_adjust(comp.p_mw.astype(float).values)
    comp['effect'] = comp.cliffs_delta.astype(float).map(eda.effect_label)
    comp['description'] = [dict(FEATURES)[f] for f in comp.index]
    save_table(comp.round(4), folder, 'train_vs_val_tests.csv')
    for f, r in comp.iterrows():
        shift = r.p_mw_holm < .05 and abs(r.cliffs_delta) >= .33
        observe('dancetrack', f'train_vs_val[{f}]', dict(delta=float(r.cliffs_delta), p_holm=float(r.p_mw_holm), effect=r.effect),
                flag='DISTRIBUTION_SHIFT' if shift else None, rule='Holm p<0.05 and |Cliff delta|>=0.33 at sequence level (train n=%d, val n=%d)' % (len(tr), len(va)))

    # ---- 3.2 val feature CIs (n sequences small) ----
    ci_rows = {f: eda.cluster_bootstrap_mean_ci(np.arange(len(va)), va[f].fillna(va[f].mean()).values, seed=SEED) for f, _ in FEATURES}
    ci_df = pd.DataFrame(ci_rows).T[['mean', 'lo', 'hi']]
    save_table(ci_df.round(4), folder, 'val_feature_means_bootstrap_ci.csv')

    # ---- 3.3 figures ----
    fig, ax = plt.subplots(2, 3, figsize=(17, 8))
    for split, col in (('train', '#1f77b4'), ('val', '#d62728')):
        p = pooled[split]
        ax[0, 0].hist(p['track_len'], bins=40, alpha=.5, density=True, color=col, label=split)
        if len(p['gap']):
            ax[0, 1].hist(np.log10(p['gap']), bins=30, alpha=.5, density=True, color=col)
        ax[0, 2].hist(np.clip(p['speed'], 0, 0.3), bins=40, alpha=.5, density=True, color=col)
        ax[1, 0].hist(p['turn'], bins=36, alpha=.5, density=True, color=col)
        ax[1, 1].hist(p['h_rel'], bins=40, alpha=.5, density=True, color=col)
        ax[1, 2].hist(p['bpf'], bins=range(0, 25), alpha=.5, density=True, color=col)
    ax[0, 0].set(title='Track length (frames present), pooled over tracks')
    ax[0, 0].legend()
    ax[0, 1].set(title='Interruption length of an ID, log10 frames')
    ax[0, 2].set(title='Speed per frame / box height (clipped 0.3)')
    ax[1, 0].set(title='Turn angle between consecutive steps (deg)')
    ax[1, 1].set(title='Box height / image height')
    ax[1, 2].set(title='Boxes per frame')
    fig.suptitle('Pooled distributions (descriptive; inference is done at sequence level)')
    savefig(fig, folder, 'pooled_distributions.png')

    fig, ax = plt.subplots(2, 3, figsize=(17, 7))
    for a, (f, desc) in zip(ax.ravel(), [FEATURES[i] for i in (0, 1, 3, 7, 9, 10)]):
        a.boxplot([tr[f].dropna(), va[f].dropna()], tick_labels=[f'train ({len(tr)})', f'val ({len(va)})'], showmeans=True)
        a.scatter(np.random.default_rng(1).normal(1, .04, len(tr)), tr[f], s=10, alpha=.5)
        a.scatter(np.random.default_rng(2).normal(2, .04, len(va)), va[f], s=10, alpha=.5)
        r = comp.loc[f]
        a.set_title(f'{desc}\nδ={r.cliffs_delta:.2f} ({r.effect}), Holm p={r.p_mw_holm:.3f}', fontsize=8)
    savefig(fig, folder, 'sequence_level_train_vs_val.png')

    corr = df[[f for f, _ in FEATURES]].corr(method='spearman')
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap='coolwarm')
    ax.set_xticks(range(len(corr)), corr.columns, rotation=60, ha='right', fontsize=7)
    ax.set_yticks(range(len(corr)), corr.columns, fontsize=7)
    ax.set_title(f'Spearman correlation of difficulty features (all {len(df)} sequences)')
    ax.grid(False)
    fig.colorbar(im)
    savefig(fig, folder, 'feature_correlation.png')
    save_table(corr.round(3), folder, 'feature_correlation.csv', show=False)

    fig, ax = plt.subplots(1, 2, figsize=(15, 4))
    for split, col in (('train', '#1f77b4'), ('val', '#d62728')):
        for (s, seq), t in timeline.items():
            if s == split:
                ax[0].plot(np.linspace(0, 1, len(t)), t, color=col, alpha=.35, lw=.8)
    ax[0].set(title='Boxes per frame over normalised time (blue train, red val)', xlabel='relative time')
    ax[1].hist([tr.cam_shift_median.dropna(), va.cam_shift_median.dropna()], bins=20, label=['train', 'val'])
    ax[1].axvline(1.0, color='k', ls='--')
    ax[1].set(title='Median camera shift between consecutive frames (px @480 wide)')
    ax[1].legend()
    savefig(fig, folder, 'density_timeline_and_camera.png')
    moving_val = int((va.cam_shift_median > 1.0).sum())
    observe('dancetrack', 'val_sequences_with_moving_camera', moving_val, flag='NO_CAMERA_STRATA' if moving_val < 5 else None,
            rule='fewer than 5 val sequences with median shift > 1 px: camera strata not analysable')
    observe('dancetrack', 'val_sequences', len(va), flag='FEW_UNITS', rule='fewer than 30 evaluation videos: wide CI, descriptive conclusions')
    observe('dancetrack', 'val_gap_missing_share_mean', float(va.gap_missing_share.mean()))

    # ---- 3.4 prior proxy check ----
    prior = pd.read_csv(next((PRIOR / 'dataset_eda' / 'dancetrack-mot').glob('*/sequence_ledger.csv')))
    m = df.merge(prior[['split', 'seq', 'camera_shift_px']], on=['split', 'seq'])
    rho = sps.spearmanr(m.cam_shift_median, m.camera_shift_px, nan_policy='omit')
    print(f'Spearman giữa proxy camera cũ (ít cặp) và mới ({CAM_PAIRS} cặp): rho={rho.statistic:.3f}')
    observe('dancetrack', 'camera_proxy_agreement_spearman', float(rho.statistic))

    # ---- 3.5 overlays: sequences spread over density, one dense/occluded frame each ----
    chosen = va.sort_values('boxes_per_frame').iloc[np.linspace(0, len(va) - 1, 6).astype(int)]
    fig, axes = plt.subplots(2, 3, figsize=(18, 9))
    for a, r in zip(axes.ravel(), chosen.itertuples()):
        e = seqs[('val', r.seq)]
        gt = gts[('val', r.seq)]
        frame = int(np.median(gt[:, 0]))
        arr = cv2.imdecode(np.frombuffer(e['zf'].read(e['frames'][frame]), np.uint8), cv2.IMREAD_COLOR)[:, :, ::-1]
        im = Image.fromarray(arr)
        d = ImageDraw.Draw(im)
        for row in gt[gt[:, 0] == frame]:
            d.rectangle((row[2], row[3], row[2] + row[4], row[3] + row[5]), outline=(255, 59, 48), width=3)
            d.text((row[2] + 3, row[3] + 3), str(int(row[1])), fill=(255, 255, 0))
        a.imshow(im)
        a.set_title(f'{r.seq} frame {frame} | {r.boxes_per_frame:.1f} box/frame | occl {r.occluded_share:.2f}', fontsize=8)
        a.axis('off')
    fig.suptitle('Val sequences spread over density, GT with identity labels')
    savefig(fig, folder, 'overlays.png')
    DT.update(df=df, gts=gts, seqs=seqs)
    return dict(sequences=len(df))


guarded('dancetrack', eda_dancetrack)

## 3b. DanceTrack — cấu trúc ID, phân tầng đóng băng, tập dev

Mục đích: hiểu vòng đời ID trước khi chạy tracker và **đóng băng phân tầng + tập dev trước khi có điểm**. Tertile tính trên **train** rồi áp sang val; không có điểm độ khó tổng hợp. Proxy camera: dịch chuyển trung vị phase-correlation 6 cặp > 1 px (@480 px rộng). Kết quả ghi `stratification-dancetrack-<date>.json` với `status=proposed`: **chỉ có hiệu lực sau khi người dùng duyệt** ngưỡng và tập dev.

In [ ]:
STRAT_AXES = {'density': 'boxes_per_frame', 'id_gaps': 'gaps_per_track', 'speed': 'speed_median'}
STRAT_DATE = '261027'
DEV_N, DEV_SEED = 8, 20261027
TIERS = ('low', 'mid', 'high')


def _font(size):
    from PIL import ImageFont
    import matplotlib
    try:
        return ImageFont.truetype(str(Path(matplotlib.get_data_path()) / 'fonts' / 'ttf' / 'DejaVuSans-Bold.ttf'), size)
    except Exception:
        return ImageFont.load_default()


def _frame_with_ids(seq_entry, gt, frame, crop):
    import cv2
    arr = cv2.imdecode(np.frombuffer(seq_entry['zf'].read(seq_entry['frames'][frame]), np.uint8), cv2.IMREAD_COLOR)[:, :, ::-1]
    im = Image.fromarray(arr)
    d = ImageDraw.Draw(im)
    font = _font(max(18, int(im.height * 0.035)))
    palette = plt.get_cmap('tab20').colors
    for row in gt[gt[:, 0] == frame]:
        col = tuple(int(255 * c) for c in palette[int(row[1]) % 20])
        d.rectangle((row[2], row[3], row[2] + row[4], row[3] + row[5]), outline=col, width=max(3, im.height // 270))
        d.text((row[2] + 4, row[3] + 2), str(int(row[1])), fill=col, font=font, stroke_width=2, stroke_fill=(0, 0, 0))
    return im.crop(crop)


def eda_dancetrack_structure():
    folder = OUT / 'dancetrack'
    df, gts, seqs = DT['df'], DT['gts'], DT['seqs']
    strata, edges = eda.stratify_sequences(df, STRAT_AXES, moving_px=1.0)
    tier_cols = [f'tier_{k}' for k in STRAT_AXES]
    # dev must lie inside the val range on every axis (feature ranges only, no scores) so tuning does not extrapolate
    val_ref = strata[strata.split == 'val']
    inside = np.all([strata[c].between(val_ref[c].min(), val_ref[c].max()) for c in STRAT_AXES.values()], axis=0)
    dev = eda.select_dev_subset(strata[(strata.split == 'train') & inside], DEV_N, DEV_SEED, tier_cols)
    strata['role'] = np.where(strata.seq.isin(dev), 'dev', strata.split)
    conf = pd.DataFrame({k: eda.id_confusion_pairs(g) for k, g in gts.items()}).T
    conf.index = pd.MultiIndex.from_tuples(conf.index, names=['split', 'seq'])
    strata = strata.join(conf, on=['split', 'seq'])
    for c in ('close_pair_frames', 'crossings'):
        strata[c + '_per_1k'] = strata[c] / strata.frames * 1000
    save_table(strata.round(4), folder, 'stratification_table.csv', show=False)
    print('Ngưỡng tertile (train):', {k: tuple(round(x, 3) for x in v) for k, v in edges.items()})
    display(strata.groupby('role').camera.value_counts().unstack(fill_value=0))
    for c in tier_cols:
        display(pd.crosstab(strata.role, strata[c]).reindex(columns=list(TIERS)))

    # ---- 3b.1 dev coverage vs val ----
    val, devdf = strata[strata.split == 'val'], strata[strata.role == 'dev']
    cols = list(STRAT_AXES.values())
    cov = eda.coverage_vs_reference(devdf, val, cols)
    save_table(cov, folder, 'dev_coverage_vs_val.csv')
    for f, r in cov.iterrows():
        observe('dancetrack', f'dev_outside_val_range[{f}]', int(r.n_outside), flag='DEV_OUT_OF_RANGE' if r.n_outside else None,
                rule='dev sequences outside the min-max of val on this axis (dev is drawn from train; change DEV_SEED if wide)')
    pairs = [('boxes_per_frame', 'gaps_per_track'), ('boxes_per_frame', 'speed_median'), ('gaps_per_track', 'speed_median')]
    axis_of = {v: k for k, v in STRAT_AXES.items()}
    rest = strata[strata.role == 'train']
    moving = strata[strata.camera == 'moving']
    fig, ax = plt.subplots(1, 3, figsize=(17, 5))
    for a, (x, y) in zip(ax, pairs):
        a.scatter(rest[x], rest[y], s=18, c='#9ecae1', label='train (rest)')
        a.scatter(val[x], val[y], s=22, c='#d62728', alpha=.7, label='val')
        a.scatter(devdf[x], devdf[y], s=80, facecolors='none', edgecolors='k', linewidths=1.6, label='dev')
        a.scatter(moving[x], moving[y], marker='x', s=40, c='#2ca02c', label='moving camera')
        for edge in edges[axis_of[x]]:
            a.axvline(edge, color='gray', ls=':')
        for edge in edges[axis_of[y]]:
            a.axhline(edge, color='gray', ls=':')
        a.set(xlabel=x, ylabel=y)
    ax[0].legend(fontsize=7)
    fig.suptitle('Dev subset vs val on the stratification axes (dotted = train tertile edges)')
    savefig(fig, folder, 'dev_coverage.png')

    # ---- 3b.2 ID lifelines, one representative val sequence per density tier ----
    reps = {}
    for t in TIERS:
        sub = val[val.tier_density == t].sort_values('boxes_per_frame')
        reps[t] = sub.iloc[len(sub) // 2].seq
    fig, ax = plt.subplots(1, 3, figsize=(18, 6))
    for a, t in zip(ax, TIERS):
        for r in eda.id_lifelines(gts[('val', reps[t])]).itertuples():
            a.broken_barh([(r.start, r.end - r.start + 1)], (r.id - .4, .8), color='#d62728' if r.occluded else '#1f77b4')
        a.set(title=f'{t} density: {reps[t]} (red = IoU>0.3 with another box)', xlabel='frame', ylabel='ID')
        a.yaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
    savefig(fig, folder, 'id_lifelines.png')

    # ---- 3b.3 interruption lengths, train vs val ----
    gl = {s: np.concatenate([eda.mot_gap_lengths(g) for (sp, _), g in gts.items() if sp == s]) for s in ('train', 'val')}
    bins = np.logspace(0, np.log10(max(max(v.max() for v in gl.values()), 2)), 30)
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    for s, col in (('train', '#1f77b4'), ('val', '#d62728')):
        ax[0].hist(gl[s], bins=bins, alpha=.5, color=col, label=f'{s} (n={len(gl[s])})')
        ax[1].hist(gl[s], bins=bins, alpha=.5, color=col, weights=gl[s], label=s)
    for a in ax:
        a.set_xscale('log')
    ax[0].set(title='Interruption length of an ID (count)', xlabel='frames')
    ax[0].legend()
    ax[1].set(title='Same, weighted by length (frames lost to interruptions)', xlabel='frames')
    savefig(fig, folder, 'interruption_lengths.png')
    for s in gl:
        observe('dancetrack', f'interruption_len[{s}]', dict(n=int(len(gl[s])), median=float(np.median(gl[s])),
                                                           p90=float(np.quantile(gl[s], .9)), max=int(gl[s].max())))

    # ---- 3b.4 typical frames around the most crowded moment, per density tier ----
    fig, axes = plt.subplots(3, 5, figsize=(25, 10))
    for row, t in zip(axes, TIERS):
        seq = reps[t]
        ov = eda.mot_frame_overlap(gts[('val', seq)])
        centre = int(ov['frames'][int(np.argmax(ov['close_pairs']))])
        length = int(strata[strata.seq == seq].frames.iloc[0])
        gt_seq = gts[('val', seq)]
        win = gt_seq[(gt_seq[:, 0] >= centre - 30) & (gt_seq[:, 0] <= centre + 30)]
        W, H = int(strata[strata.seq == seq].width.iloc[0]), int(strata[strata.seq == seq].height.iloc[0])
        mx, my = .08 * (win[:, 2] + win[:, 4]).max(), .08 * (win[:, 3] + win[:, 5]).max()
        crop = (int(max(0, win[:, 2].min() - mx)), int(max(0, win[:, 3].min() - my)),
                int(min(W, (win[:, 2] + win[:, 4]).max() + mx)), int(min(H, (win[:, 3] + win[:, 5]).max() + my)))   # same crop for the strip, so IDs stay legible
        for a, off in zip(row, (-30, -15, 0, 15, 30)):
            fr = int(np.clip(centre + off, 1, length))
            a.imshow(_frame_with_ids(seqs[('val', seq)], gt_seq, fr, crop))
            a.set_title(f'{t} | {seq} frame {fr}', fontsize=9)
            a.axis('off')
    fig.suptitle('Frames around the most crowded moment (box and ID colour = GT identity)')
    savefig(fig, folder, 'frames_by_tier.png')

    # ---- 3b.5 close / crossing pairs ----
    rho = sps.spearmanr(strata.crossings_per_1k, strata.gaps_per_track)
    observe('dancetrack', 'crossings_vs_gaps_spearman', float(rho.statistic))
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    for a, c in zip(ax, ('close_pair_frames_per_1k', 'crossings_per_1k')):
        a.scatter(strata.boxes_per_frame, strata[c], c=np.where(strata.split == 'val', '#d62728', '#1f77b4'), s=20)
        a.set(xlabel='boxes_per_frame', ylabel=c, title=c)
    savefig(fig, folder, 'confusable_pairs.png')

    # ---- 3b.6 frozen proposal (status=proposed until the user approves) ----
    payload = dict(
        status='proposed', approved_by=None, created=datetime.now(timezone.utc).isoformat(),
        binds_registry_sequence_ids_sha256=REG['mot']['sequence_ids_sha256'],
        rule=dict(tiers='low: v<=e1, mid: e1<v<=e2, high: v>e2; e1,e2 = 1/3 and 2/3 quantiles of the 40 TRAIN sequences, applied unchanged to val',
                  axes=STRAT_AXES, composite_score=None,
                  camera_proxy='median phase-correlation shift over %d consecutive-frame pairs at 480 px width > 1.0 px => moving' % CAM_PAIRS),
        edges={k: list(v) for k, v in edges.items()},
        dev=dict(seed=DEV_SEED, n=DEV_N, source_split='train', sequences=dev,
                 constraint='inside the val min-max on every axis; each tier of each axis >= 2 sequences; >= 1 moving camera'),
        counts=dict(camera_moving={s: int(((strata.split == s) & (strata.camera == 'moving')).sum()) for s in ('train', 'val')}),
        sequences=[dict(split=r.split, seq=r.seq, role=r.role, camera=r.camera, **{c: getattr(r, c) for c in tier_cols})
                   for r in strata.sort_values(['split', 'seq']).itertuples()])
    write_json(OUT / f'stratification-dancetrack-{STRAT_DATE}.json', payload)
    observe('dancetrack', 'dev_sequences', dev)
    observe('dancetrack', 'moving_camera_sequences', moving.seq.tolist(), flag='CAMERA_DESCRIPTIVE_ONLY',
            rule='3 or fewer moving val sequences: camera results are descriptive, no conclusion')
    return dict(dev=dev, edges=edges)


guarded('dancetrack_structure', eda_dancetrack_structure)

## 4. Bảng quan sát và gói kết quả

Mỗi cờ là một **quy tắc heuristic ghi rõ trong cột `rule`**, không phải kết luận. Cờ chỉ ra chỗ phase 3/4 cần xử lý (báo cáo riêng lớp ít hỗ trợ, bootstrap theo nhóm, không phân tầng theo camera, ...).

In [ ]:
obs = pd.DataFrame(OBS)
display(obs[obs.flag.notna()][['dataset', 'key', 'flag', 'value', 'rule']].to_string(index=False) if obs.flag.notna().any() else 'Không có cờ.')
write_json(OUT / 'eda-observations.json', dict(
    run_id=RUN_ID, finished_utc=datetime.now(timezone.utc).isoformat(), python=platform.python_version(), seed=SEED,
    inputs_verified=dict(lvvo_gt=REG['lvvo']['gt_coco_canonical_sha256'], coco_gt=REG['coco']['gt_coco_canonical_sha256'],
                         mot_sequences_sha=REG['mot']['sequence_ids_sha256']),
    sections={k: ('ERROR' if v.get('error') else 'ok') for k, v in RESULTS.items()}, observations=OBS,
    model_inference_run=False, scoring_run=False))
bundle = OUT / 'bundle.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUT.rglob('*')):
        if path.is_file() and path != bundle and path.suffix in {'.json', '.csv', '.png'}:
            zf.write(path, path.relative_to(OUT))
print('Bundle:', bundle, f'{bundle.stat().st_size / 1e6:.1f} MB', '| sections:', {k: ('ERROR' if v.get('error') else 'ok') for k, v in RESULTS.items()})